# ANKA-50M RMW3 — Full 5B Training and Official Evaluation

> **Status:** Released base-model training notebook

## Purpose

Load the final English corpus into RAM, define the exact 48,944,657-parameter RMW3 architecture, train from random initialization for 5B tokens, and run the official evaluations.

## Execution guidance

Use a fresh A100 80 GB runtime. Execute setup, RAM loading, model definition, and the single full-training cell in order. The evaluation section is run after the final checkpoint exists. Stage-2 cells are preserved as later experiments and are not part of the released base checkpoint.

## Primary outputs

The released final.pt checkpoint, training history, 5B-token summary, four zero-shot benchmark reports, WikiText-103 perplexity, and generation samples.

## Released-run efficiency

| Measurement | Value |
|---|---:|
| Trainable parameters | 48,944,657 |
| Training tokens | 5,000,000,000 |
| Hardware | 1 × NVIDIA A100-SXM4 80 GB |
| Average throughput | 117,899 tokens/s |
| Wall time | 11 h 46 m 49 s |
| Compute | 11.78 GPU-hours |
| Tokens per GPU-hour | 424.4M |
| Best internal validation loss | 2.334379 |

## Reproducibility notes

- Code-cell outputs and execution counters are intentionally cleared in the public release.
- Seeds, persistent paths, and checkpoint-resume behavior remain in the code.
- Review path constants before execution; historical notebooks may reference earlier run directories.
- Execute sections from top to bottom unless a section explicitly describes an independent experiment arm.


## 1. Set up Drive, paths, and deterministic execution

This is a compute-intensive stage. Confirm the selected runtime, input paths, token budget, and resume directory before execution. Checkpoints are written incrementally.


In [ ]:
# ============================================================
# 1 — SETUP AND DRIVE CHECK
# ============================================================

%pip install -q "tokenizers>=0.15" "psutil>=5.9"

from google.colab import drive

drive.mount(
    "/content/drive",
    force_remount=False,
)


import gc
import json
import os
import random
from pathlib import Path

import numpy as np
import psutil
import torch


SEED = 3407

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


RUN_ROOT = Path(
    "/content/drive/MyDrive/ANKA/"
    "run_02_english_5b"
)

TOKENIZER_PATH = (
    RUN_ROOT
    / "tokenizer"
    / "tokenizer.json"
)

NATURAL_MANIFEST_PATH = (
    RUN_ROOT
    / "natural_4p8b_v3"
    / "manifest.json"
)

WORLD_MANIFEST_PATH = (
    RUN_ROOT
    / "worldpairs_200m_v3"
    / "manifest.json"
)

TRAIN_ROOT = (
    RUN_ROOT
    / "rmw3_full_5b_v3"
)

TRAIN_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


required_paths = [
    TOKENIZER_PATH,
    NATURAL_MANIFEST_PATH,
    WORLD_MANIFEST_PATH,
]

for required_path in required_paths:
    if not required_path.exists():
        raise FileNotFoundError(
            f"missing file: {required_path}"
        )


natural_manifest = json.loads(
    NATURAL_MANIFEST_PATH.read_text(
        encoding="utf-8"
    )
)

world_manifest = json.loads(
    WORLD_MANIFEST_PATH.read_text(
        encoding="utf-8"
    )
)


natural_tokens = sum(
    int(item["tokens"])
    for item
    in natural_manifest["train_shards"]
)

validation_tokens = sum(
    int(item["tokens"])
    for item
    in natural_manifest["validation_shards"]
)

world_tokens = sum(
    int(item["tokens"])
    for item
    in world_manifest["shards"]
)


if natural_tokens != 4_800_000_000:
    raise RuntimeError(
        f"Incorrect natural token count: "
        f"{natural_tokens:,}"
    )

if world_tokens != 200_000_000:
    raise RuntimeError(
        f"Incorrect World token count: "
        f"{world_tokens:,}"
    )

if validation_tokens != 8_000_000:
    raise RuntimeError(
        f"Incorrect validation token count: "
        f"{validation_tokens:,}"
    )


missing_shards = []

for item in (
    natural_manifest["train_shards"]
    + natural_manifest["validation_shards"]
    + world_manifest["shards"]
):
    shard_path = Path(
        item["path"]
    )

    if not shard_path.exists():
        missing_shards.append(
            str(shard_path)
        )


if missing_shards:
    raise FileNotFoundError(
        "missing shard found:\n"
        + "\n".join(
            missing_shards[:10]
        )
    )


available_ram_gib = (
    psutil.virtual_memory().available
    / 2**30
)

print("ALL DRIVE FILES ARE READY")
print("Natural    :", f"{natural_tokens:,}")
print("WorldPairs :", f"{world_tokens:,}")
print("Validation :", f"{validation_tokens:,}")
print(
    "Total     :",
    f"{natural_tokens + world_tokens:,}",
)
print(
    "Free RAM    :",
    f"{available_ram_gib:.2f} GiB",
)
print("Training path :", TRAIN_ROOT)

## 2. Load the 5B-token training corpus and validation stream into RAM

This is a compute-intensive stage. Confirm the selected runtime, input paths, token budget, and resume directory before execution. Checkpoints are written incrementally.


In [ ]:
# ============================================================
# 2 — LOAD DRIVE CORPUS TO RAM
# ============================================================

import gc
import json
import math
from concurrent.futures import (
    Future,
    ThreadPoolExecutor,
)
from pathlib import Path

import numpy as np
import torch


SEQ_LEN = 1024
DATA_SEED = 3407


# Drops old RAM arrays first if cell is accidentally run twice.
for variable_name in (
    "NATURAL_RAM",
    "WORLD_RAM",
    "VALIDATION_RAM",
):
    if variable_name in globals():
        del globals()[variable_name]

gc.collect()


natural_manifest = json.loads(
    NATURAL_MANIFEST_PATH.read_text(
        encoding="utf-8"
    )
)

world_manifest = json.loads(
    WORLD_MANIFEST_PATH.read_text(
        encoding="utf-8"
    )
)


def load_uint16_shards(
    shard_items: list[dict],
    label: str,
) -> np.ndarray:
    shard_paths = []
    shard_sizes = []

    for item in shard_items:
        path = Path(
            item["path"]
        )

        if not path.exists():
            raise FileNotFoundError(
                path
            )

        actual_tokens = (
            path.stat().st_size
            // np.dtype(
                np.uint16
            ).itemsize
        )

        expected_tokens = int(
            item["tokens"]
        )

        if actual_tokens != expected_tokens:
            raise RuntimeError(
                f"Corrupted shard: {path}\n"
                f"Expected: {expected_tokens:,}\n"
                f"Actual: {actual_tokens:,}"
            )

        shard_paths.append(path)
        shard_sizes.append(
            actual_tokens
        )

    total_tokens = sum(
        shard_sizes
    )

    output = np.empty(
        total_tokens,
        dtype=np.uint16,
    )

    cursor = 0

    for shard_number, (
        path,
        shard_size,
    ) in enumerate(
        zip(
            shard_paths,
            shard_sizes,
        ),
        start=1,
    ):
        mapped = np.memmap(
            path,
            dtype=np.uint16,
            mode="r",
            shape=(shard_size,),
        )

        output[
            cursor:
            cursor + shard_size
        ] = mapped

        cursor += shard_size

        del mapped

        if (
            shard_number % 10 == 0
            or
            shard_number
            == len(shard_paths)
        ):
            print(
                f"{label}: "
                f"{cursor:,}/"
                f"{total_tokens:,} "
                "tokens loaded to RAM"
            )

    return output


print(
    "Loading natural corpus to RAM..."
)

NATURAL_RAM = load_uint16_shards(
    natural_manifest["train_shards"],
    "natural",
)

print(
    "Loading WorldPairs to RAM..."
)

WORLD_RAM = load_uint16_shards(
    world_manifest["shards"],
    "world",
)

print(
    "Loading validation to RAM..."
)

VALIDATION_RAM = load_uint16_shards(
    natural_manifest[
        "validation_shards"
    ],
    "validation",
)


if (
    len(NATURAL_RAM)
    != 4_800_000_000
):
    raise RuntimeError(
        "Natural RAM token count is incorrect."
    )

if (
    len(WORLD_RAM)
    != 200_000_000
):
    raise RuntimeError(
        "World RAM token count is incorrect."
    )

if (
    len(VALIDATION_RAM)
    != 8_000_000
):
    raise RuntimeError(
        "Validation RAM token count is incorrect."
    )


maximum_token_id = max(
    int(NATURAL_RAM.max()),
    int(WORLD_RAM.max()),
    int(VALIDATION_RAM.max()),
)

if maximum_token_id >= 16_384:
    raise RuntimeError(
        f"Invalid token ID: "
        f"{maximum_token_id}"
    )


NATURAL_ROWS = (
    (len(NATURAL_RAM) - 1)
    // SEQ_LEN
)

WORLD_ROWS = (
    (len(WORLD_RAM) - 1)
    // SEQ_LEN
)


class ExactRamBatcher:
    """
    Deterministically shuffles Natural and WorldPairs rows.

    If training is interrupted, returns to the same position via seen_tokens.
    """

    def __init__(
        self,
        natural_tokens: np.ndarray,
        world_tokens: np.ndarray,
        seed: int = DATA_SEED,
    ):
        self.natural_tokens = (
            natural_tokens
        )

        self.world_tokens = (
            world_tokens
        )

        random_generator = (
            np.random.default_rng(
                seed
            )
        )

        self.natural_order = (
            random_generator
            .permutation(
                NATURAL_ROWS
            )
            .astype(
                np.int32,
                copy=False,
            )
        )

        self.world_order = (
            random_generator
            .permutation(
                WORLD_ROWS
            )
            .astype(
                np.int32,
                copy=False,
            )
        )

        self.schedule = np.concatenate(
            (
                np.zeros(
                    NATURAL_ROWS,
                    dtype=np.uint8,
                ),
                np.ones(
                    WORLD_ROWS,
                    dtype=np.uint8,
                ),
            )
        )

        random_generator.shuffle(
            self.schedule
        )

        self.schedule_position = 0
        self.natural_position = 0
        self.world_position = 0

    def seek_tokens(
        self,
        seen_tokens: int,
    ) -> None:
        consumed_rows = (
            int(seen_tokens)
            // SEQ_LEN
        )

        schedule_length = len(
            self.schedule
        )

        completed_cycles, remainder = (
            divmod(
                consumed_rows,
                schedule_length,
            )
        )

        schedule_prefix = (
            self.schedule[:remainder]
        )

        self.schedule_position = (
            consumed_rows
        )

        self.natural_position = (
            completed_cycles
            * NATURAL_ROWS
            + int(
                (
                    schedule_prefix
                    == 0
                ).sum()
            )
        )

        self.world_position = (
            completed_cycles
            * WORLD_ROWS
            + int(
                (
                    schedule_prefix
                    == 1
                ).sum()
            )
        )

    def _take_natural_row(
        self,
    ) -> np.ndarray:
        row_index = int(
            self.natural_order[
                self.natural_position
                % NATURAL_ROWS
            ]
        )

        self.natural_position += 1

        start = (
            row_index
            * SEQ_LEN
        )

        return self.natural_tokens[
            start:
            start + SEQ_LEN + 1
        ]

    def _take_world_row(
        self,
    ) -> np.ndarray:
        row_index = int(
            self.world_order[
                self.world_position
                % WORLD_ROWS
            ]
        )

        self.world_position += 1

        start = (
            row_index
            * SEQ_LEN
        )

        return self.world_tokens[
            start:
            start + SEQ_LEN + 1
        ]

    def make_batch(
        self,
        maximum_batch_size: int,
        active_token_count: int,
    ):
        rows_needed = min(
            maximum_batch_size,
            math.ceil(
                active_token_count
                / SEQ_LEN
            ),
        )

        packed = np.empty(
            (
                rows_needed,
                SEQ_LEN + 1,
            ),
            dtype=np.int64,
        )

        world_rows_in_batch = 0

        for batch_row in range(
            rows_needed
        ):
            row_type = int(
                self.schedule[
                    self.schedule_position
                    % len(self.schedule)
                ]
            )

            self.schedule_position += 1

            if row_type == 0:
                packed[
                    batch_row
                ] = (
                    self._take_natural_row()
                )

            else:
                packed[
                    batch_row
                ] = (
                    self._take_world_row()
                )

                world_rows_in_batch += 1

        inputs = torch.from_numpy(
            packed[:, :-1].copy()
        ).pin_memory()

        labels = torch.from_numpy(
            packed[:, 1:].copy()
        ).pin_memory()

        valid_tokens = min(
            active_token_count,
            rows_needed * SEQ_LEN,
        )

        if (
            valid_tokens
            < rows_needed * SEQ_LEN
        ):
            labels.view(-1)[
                valid_tokens:
            ] = -100

        world_ratio = (
            world_rows_in_batch
            / rows_needed
        )

        return (
            inputs,
            labels,
            valid_tokens,
            world_ratio,
        )


class PrefetchRamLoader:
    """
    Single CPU worker keeps the data order deterministic.
    Two batches are prepared in advance.
    """

    def __init__(
        self,
        batcher: ExactRamBatcher,
        batch_size: int,
        target_tokens: int,
        start_tokens: int,
    ):
        self.batcher = batcher
        self.batch_size = batch_size

        self.target_tokens = (
            target_tokens
        )

        self.submitted_tokens = (
            start_tokens
        )

        self.executor = (
            ThreadPoolExecutor(
                max_workers=1
            )
        )

        self.queue: list[Future] = []

        self._fill_queue()

    def _fill_queue(
        self,
    ) -> None:
        while (
            len(self.queue) < 2
            and
            self.submitted_tokens
            < self.target_tokens
        ):
            remaining_tokens = (
                self.target_tokens
                - self.submitted_tokens
            )

            active_tokens = min(
                remaining_tokens,
                self.batch_size
                * SEQ_LEN,
            )

            future = self.executor.submit(
                self.batcher.make_batch,
                self.batch_size,
                active_tokens,
            )

            self.queue.append(
                future
            )

            self.submitted_tokens += (
                active_tokens
            )

    def __iter__(
        self,
    ):
        return self

    def __next__(
        self,
    ):
        if not self.queue:
            raise StopIteration

        result = (
            self.queue
            .pop(0)
            .result()
        )

        self._fill_queue()

        return result

    def close(
        self,
    ) -> None:
        self.executor.shutdown(
            wait=False,
            cancel_futures=True,
        )


print("\nRAM DATA READY")
print(
    "Natural token :",
    f"{len(NATURAL_RAM):,}",
)
print(
    "World token   :",
    f"{len(WORLD_RAM):,}",
)
print(
    "Validation    :",
    f"{len(VALIDATION_RAM):,}",
)
print(
    "Total train   :",
    f"{len(NATURAL_RAM) + len(WORLD_RAM):,}",
)

## 3. Define the exact RMW3 architecture

Defines or validates model components used by the experiment. Keep tensor shapes, parameter counts, and causal behavior unchanged when reproducing the run.


In [ ]:
# # ============================================================
# 3 — RMW3 MODEL definition
# ============================================================

from dataclasses import dataclass
from typing import Optional

import math
import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class RMW3Config:
    vocab_size: int = 16_384
    d_model: int = 512

    n_physical_layers: int = 13
    recurrent_passes: int = 2

    n_heads: int = 8
    n_kv_heads: int = 2

    d_ff: int = 1_280

    reflect_rank: int = 32

    morph_rank: int = 368
    morph_stages: int = 4
    morph_experts: int = 4

    max_seq_len: int = 1_024

    rope_theta: float = 10_000.0
    norm_eps: float = 1e-5


class RMSNorm(nn.Module):
    def __init__(
        self,
        dimension: int,
        epsilon: float,
    ):
        super().__init__()

        self.weight = nn.Parameter(
            torch.ones(dimension)
        )

        self.epsilon = epsilon

    def forward(
        self,
        x: torch.Tensor,
    ) -> torch.Tensor:
        inverse_rms = torch.rsqrt(
            x.float()
            .pow(2)
            .mean(
                dim=-1,
                keepdim=True,
            )
            + self.epsilon
        )

        normalized = (
            x.float()
            * inverse_rms
        ).to(x.dtype)

        return (
            normalized
            * self.weight
        )


def build_rope_cache(
    maximum_sequence_length: int,
    head_dimension: int,
    theta: float,
):
    inverse_frequency = 1.0 / (
        theta ** (
            torch.arange(
                0,
                head_dimension,
                2,
                dtype=torch.float32,
            )
            / head_dimension
        )
    )

    positions = torch.arange(
        maximum_sequence_length,
        dtype=torch.float32,
    )

    angles = torch.outer(
        positions,
        inverse_frequency,
    )

    return (
        angles.cos(),
        angles.sin(),
    )


def apply_rope(
    x: torch.Tensor,
    cosine: torch.Tensor,
    sine: torch.Tensor,
) -> torch.Tensor:
    sequence_length = x.shape[-2]

    cosine = cosine[
        :sequence_length
    ].to(
        device=x.device,
        dtype=x.dtype,
    )[None, None, :, :]

    sine = sine[
        :sequence_length
    ].to(
        device=x.device,
        dtype=x.dtype,
    )[None, None, :, :]

    even = x[..., 0::2]
    odd = x[..., 1::2]

    rotated = torch.stack(
        (
            even * cosine
            - odd * sine,
            even * sine
            + odd * cosine,
        ),
        dim=-1,
    )

    return rotated.flatten(-2)


class GroupedQueryAttention(nn.Module):
    def __init__(
        self,
        config: RMW3Config,
    ):
        super().__init__()

        self.number_of_heads = (
            config.n_heads
        )

        self.number_of_kv_heads = (
            config.n_kv_heads
        )

        self.head_dimension = (
            config.d_model
            // config.n_heads
        )

        assert (
            config.d_model
            % config.n_heads
            == 0
        )

        assert (
            config.n_heads
            % config.n_kv_heads
            == 0
        )

        self.query_projection = nn.Linear(
            config.d_model,
            config.n_heads
            * self.head_dimension,
            bias=False,
        )

        self.key_projection = nn.Linear(
            config.d_model,
            config.n_kv_heads
            * self.head_dimension,
            bias=False,
        )

        self.value_projection = nn.Linear(
            config.d_model,
            config.n_kv_heads
            * self.head_dimension,
            bias=False,
        )

        self.output_projection = nn.Linear(
            config.n_heads
            * self.head_dimension,
            config.d_model,
            bias=False,
        )

        cosine, sine = build_rope_cache(
            config.max_seq_len,
            self.head_dimension,
            config.rope_theta,
        )

        self.register_buffer(
            "rope_cosine",
            cosine,
            persistent=False,
        )

        self.register_buffer(
            "rope_sine",
            sine,
            persistent=False,
        )

    def forward(
        self,
        x: torch.Tensor,
    ) -> torch.Tensor:
        batch_size = x.shape[0]
        sequence_length = x.shape[1]

        query = (
            self.query_projection(x)
            .view(
                batch_size,
                sequence_length,
                self.number_of_heads,
                self.head_dimension,
            )
            .transpose(1, 2)
        )

        key = (
            self.key_projection(x)
            .view(
                batch_size,
                sequence_length,
                self.number_of_kv_heads,
                self.head_dimension,
            )
            .transpose(1, 2)
        )

        value = (
            self.value_projection(x)
            .view(
                batch_size,
                sequence_length,
                self.number_of_kv_heads,
                self.head_dimension,
            )
            .transpose(1, 2)
        )

        query = apply_rope(
            query,
            self.rope_cosine,
            self.rope_sine,
        )

        key = apply_rope(
            key,
            self.rope_cosine,
            self.rope_sine,
        )

        output = (
            F.scaled_dot_product_attention(
                query,
                key,
                value,
                dropout_p=0.0,
                is_causal=True,
                enable_gqa=True,
            )
        )

        output = (
            output
            .transpose(1, 2)
            .contiguous()
            .view(
                batch_size,
                sequence_length,
                -1,
            )
        )

        return self.output_projection(
            output
        )


class SwiGLU(nn.Module):
    def __init__(
        self,
        config: RMW3Config,
    ):
        super().__init__()

        self.gate_projection = nn.Linear(
            config.d_model,
            config.d_ff,
            bias=False,
        )

        self.up_projection = nn.Linear(
            config.d_model,
            config.d_ff,
            bias=False,
        )

        self.down_projection = nn.Linear(
            config.d_ff,
            config.d_model,
            bias=False,
        )

    def forward(
        self,
        x: torch.Tensor,
    ) -> torch.Tensor:
        return self.down_projection(
            F.silu(
                self.gate_projection(x)
            )
            * self.up_projection(x)
        )


class RMW3Block(nn.Module):
    def __init__(
        self,
        config: RMW3Config,
    ):
        super().__init__()

        self.attention_norm = RMSNorm(
            config.d_model,
            config.norm_eps,
        )

        self.attention = (
            GroupedQueryAttention(
                config
            )
        )

        self.ffn_norm = RMSNorm(
            config.d_model,
            config.norm_eps,
        )

        self.ffn = SwiGLU(
            config
        )

    def forward(
        self,
        x: torch.Tensor,
    ) -> torch.Tensor:
        x = (
            x
            + self.attention(
                self.attention_norm(x)
            )
        )

        x = (
            x
            + self.ffn(
                self.ffn_norm(x)
            )
        )

        return x


class ReflectUnit(nn.Module):
    def __init__(
        self,
        config: RMW3Config,
    ):
        super().__init__()

        self.down_projection = nn.Linear(
            config.d_model,
            config.reflect_rank,
            bias=False,
        )

        self.up_projection = nn.Linear(
            config.reflect_rank,
            config.d_model,
            bias=False,
        )

        self.gate_logit = nn.Parameter(
            torch.tensor(
                -2.1972246,
                dtype=torch.float32,
            )
        )

    def forward(
        self,
        current: torch.Tensor,
        previous: torch.Tensor,
    ) -> torch.Tensor:
        discrepancy = (
            current
            - previous
        )

        correction = (
            self.up_projection(
                F.silu(
                    self.down_projection(
                        discrepancy
                    )
                )
            )
        )

        gate = torch.sigmoid(
            self.gate_logit
        ).to(current.dtype)

        return (
            current
            + gate * correction
        )


class MorphExpert(nn.Module):
    def __init__(
        self,
        config: RMW3Config,
    ):
        super().__init__()

        self.down_projection = nn.Linear(
            config.d_model,
            config.morph_rank,
            bias=False,
        )

        self.up_projection = nn.Linear(
            config.morph_rank,
            config.d_model,
            bias=False,
        )

    def forward(
        self,
        x: torch.Tensor,
    ) -> torch.Tensor:
        return self.up_projection(
            F.silu(
                self.down_projection(x)
            )
        )


class SparseMorphStage(nn.Module):
    def __init__(
        self,
        config: RMW3Config,
    ):
        super().__init__()

        self.number_of_experts = (
            config.morph_experts
        )

        self.router = nn.Linear(
            config.d_model,
            config.morph_experts,
            bias=False,
        )

        self.experts = nn.ModuleList(
            [
                MorphExpert(config)
                for _ in range(
                    config.morph_experts
                )
            ]
        )

        self.scale_logit = nn.Parameter(
            torch.tensor(
                -4.0,
                dtype=torch.float32,
            )
        )

        self.last_loads: Optional[
            torch.Tensor
        ] = None

        self.last_entropy: Optional[
            torch.Tensor
        ] = None

    def forward(
        self,
        x: torch.Tensor,
    ):
        original_shape = x.shape

        flat = x.reshape(
            -1,
            original_shape[-1],
        )

        probabilities = torch.softmax(
            self.router(flat).float(),
            dim=-1,
        )

        choices = probabilities.argmax(
            dim=-1
        )

        output = torch.zeros_like(
            flat
        )

        loads = []

        for expert_index, expert in enumerate(
            self.experts
        ):
            selected_mask = (
                choices
                == expert_index
            )

            loads.append(
                selected_mask
                .float()
                .mean()
            )

            indices = torch.nonzero(
                selected_mask,
                as_tuple=False,
            ).flatten()

            if indices.numel() == 0:
                continue

            selected_input = (
                flat.index_select(
                    0,
                    indices,
                )
            )

            expert_output = expert(
                selected_input
            ).to(output.dtype)

            selected_probability = (
                probabilities
                .index_select(
                    0,
                    indices,
                )
                [:, expert_index]
                .to(output.dtype)
            )

            straight_through = (
                1.0
                + selected_probability
                - selected_probability.detach()
            )

            output.index_copy_(
                0,
                indices,
                (
                    expert_output
                    * straight_through.unsqueeze(-1)
                ),
            )

        load_tensor = torch.stack(
            loads
        )

        importance = probabilities.mean(
            dim=0
        )

        balance_loss = (
            self.number_of_experts
            * torch.sum(
                load_tensor
                * importance
            )
        )

        entropy = -(
            probabilities
            * probabilities
            .clamp_min(1e-9)
            .log()
        ).sum(
            dim=-1
        ).mean()

        self.last_loads = (
            load_tensor.detach()
        )

        self.last_entropy = (
            entropy.detach()
        )

        scale = torch.sigmoid(
            self.scale_logit
        ).to(output.dtype)

        return (
            x
            + scale
            * output.view(
                original_shape
            ),
            balance_loss,
        )


class RMW3(nn.Module):
    def __init__(
        self,
        config: RMW3Config,
    ):
        super().__init__()

        self.config = config

        self.token_embeddings = nn.Embedding(
            config.vocab_size,
            config.d_model,
        )

        self.blocks = nn.ModuleList(
            [
                RMW3Block(config)
                for _ in range(
                    config.n_physical_layers
                )
            ]
        )

        self.reflect_units = nn.ModuleList(
            [
                ReflectUnit(config)
                for _ in range(
                    config.n_physical_layers
                )
            ]
        )

        self.morph_stages = nn.ModuleList(
            [
                SparseMorphStage(config)
                for _ in range(
                    config.morph_stages
                )
            ]
        )

        self.final_norm = RMSNorm(
            config.d_model,
            config.norm_eps,
        )

        self.lm_head = nn.Linear(
            config.d_model,
            config.vocab_size,
            bias=False,
        )

        self.apply(
            self._initialize_weights
        )

        self.lm_head.weight = (
            self.token_embeddings.weight
        )

        self._scale_residual_outputs()

    @staticmethod
    def _initialize_weights(
        module: nn.Module,
    ) -> None:
        if isinstance(
            module,
            (
                nn.Linear,
                nn.Embedding,
            ),
        ):
            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02,
            )

    def _scale_residual_outputs(
        self,
    ) -> None:
        residual_scale = (
            1.0
            / math.sqrt(
                2
                * self.config.n_physical_layers
                * self.config.recurrent_passes
            )
        )

        with torch.no_grad():
            for block in self.blocks:
                block.attention.output_projection.weight.mul_(
                    residual_scale
                )

                block.ffn.down_projection.weight.mul_(
                    residual_scale
                )

            for reflect_unit in (
                self.reflect_units
            ):
                reflect_unit.up_projection.weight.mul_(
                    residual_scale
                )

            for stage in (
                self.morph_stages
            ):
                for expert in (
                    stage.experts
                ):
                    expert.up_projection.weight.mul_(
                        residual_scale
                    )

    def forward(
        self,
        input_ids: torch.Tensor,
    ):
        if input_ids.ndim != 2:
            raise ValueError(
                "input_ids must be two-dimensional."
            )

        if (
            input_ids.shape[1]
            > self.config.max_seq_len
        ):
            raise ValueError(
                "Sequence length limit exceeded."
            )

        x = self.token_embeddings(
            input_ids
        )

        first_pass_states = []
        balance_losses = []

        effective_layer_index = 0

        total_effective_layers = (
            self.config.n_physical_layers
            * self.config.recurrent_passes
        )

        for recurrent_pass in range(
            self.config.recurrent_passes
        ):
            for layer_index, block in enumerate(
                self.blocks
            ):
                if recurrent_pass == 1:
                    x = self.reflect_units[
                        layer_index
                    ](
                        x,
                        first_pass_states[
                            layer_index
                        ],
                    )

                x = block(x)

                if recurrent_pass == 0:
                    first_pass_states.append(
                        x
                    )

                current_stage = min(
                    self.config.morph_stages - 1,
                    (
                        effective_layer_index
                        * self.config.morph_stages
                        // total_effective_layers
                    ),
                )

                next_stage = min(
                    self.config.morph_stages - 1,
                    (
                        (
                            effective_layer_index
                            + 1
                        )
                        * self.config.morph_stages
                        // total_effective_layers
                    ),
                )

                stage_finished = (
                    next_stage
                    != current_stage
                    or
                    effective_layer_index
                    == total_effective_layers - 1
                )

                if stage_finished:
                    (
                        x,
                        balance_loss,
                    ) = self.morph_stages[
                        current_stage
                    ](x)

                    balance_losses.append(
                        balance_loss
                    )

                effective_layer_index += 1

        logits = self.lm_head(
            self.final_norm(x)
        )

        auxiliary_loss = (
            torch.stack(
                balance_losses
            ).mean()
            if balance_losses
            else logits.new_zeros(())
        )

        return (
            logits,
            auxiliary_loss,
        )


RMW3_CONFIG = RMW3Config()

parameter_probe = RMW3(
    RMW3_CONFIG
)

RMW3_PARAMETERS = sum(
    parameter.numel()
    for parameter
    in parameter_probe.parameters()
)

if RMW3_PARAMETERS > 50_000_000:
    raise RuntimeError(
        f"50M parameter limit exceeded: "
        f"{RMW3_PARAMETERS:,}"
    )

if (
    parameter_probe
    .lm_head
    .weight
    .data_ptr()
    !=
    parameter_probe
    .token_embeddings
    .weight
    .data_ptr()
):
    raise RuntimeError(
        "Weight tying is not active."
    )


print("RMW3 MODEL ready")
print(
    "parameter          :",
    f"{RMW3_PARAMETERS:,}",
)
print(
    "physical/efektif   :",
    RMW3_CONFIG.n_physical_layers,
    "/",
    (
        RMW3_CONFIG.n_physical_layers
        * RMW3_CONFIG.recurrent_passes
    ),
)
print(
    "Vocabulary         :",
    RMW3_CONFIG.vocab_size,
)
print(
    "Sequence           :",
    RMW3_CONFIG.max_seq_len,
)

del parameter_probe
gc.collect()

## 4. Train the released 5B-token base model

This is a compute-intensive stage. Confirm the selected runtime, input paths, token budget, and resume directory before execution. Checkpoints are written incrementally.

> **Post-release change:** this cell compiles the forward pass and loss with `torch.compile` (`USE_TORCH_COMPILE = True`). The math is unchanged, but the released 5B run used eager mode; set `USE_TORCH_COMPILE = False` to reproduce it exactly.


In [ ]:
# ============================================================
# FINAL TRAINING CELL — RMW3 full 5B training
# verified safe batch size = 64
# ============================================================

import gc
import inspect
import json
import math
import os
import random
import shutil
import sys
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F


# ============================================================
# 1. CLEAR GPU REFERENCES LEFT BY THE PREVIOUS OOM
# ============================================================

if "loader" in globals():
    try:
        loader.close()
    except Exception:
        pass


cleanup_names = (
    "model",
    "optimizer",
    "loader",
    "batcher",
    "inputs",
    "labels",
    "cpu_inputs",
    "cpu_labels",
    "total_loss",
    "language_loss",
    "balance_loss",
    "gradient_norm",
    "test_model",
    "test_optimizer",
    "test_tokens",
    "test_loss",
    "checkpoint",
    "compiled_training_loss",
)

for cleanup_name in cleanup_names:
    if cleanup_name in globals():
        globals()[cleanup_name] = None

# Compiled graphs from an earlier run of this cell hold references
# to the old model's parameters.
torch._dynamo.reset()

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    torch.cuda.synchronize()

sys.last_traceback = None
sys.last_value = None
sys.last_type = None

gc.collect()
torch.cuda.empty_cache()


# ============================================================
# 2. VERIFY REQUIRED OBJECTS
# ============================================================

required_names = (
    "RMW3",
    "RMW3_CONFIG",
    "RMW3_PARAMETERS",
    "ExactRamBatcher",
    "PrefetchRamLoader",
    "NATURAL_RAM",
    "WORLD_RAM",
    "VALIDATION_RAM",
    "SEQ_LEN",
    "DATA_SEED",
    "TRAIN_ROOT",
    "SEED",
)

missing_names = [
    name
    for name in required_names
    if name not in globals()
]

if missing_names:
    raise RuntimeError(
        "missing notebook definitions: "
        + ", ".join(missing_names)
    )


if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU not found."
    )

if not torch.cuda.is_bf16_supported():
    raise RuntimeError(
        "BF16 capable GPU required."
    )


# ============================================================
# 3. TRAINING SETTINGS
# ============================================================

DEVICE = torch.device("cuda")

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

torch.set_float32_matmul_precision(
    "high"
)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)


TRAIN_ROOT = Path(
    "/content/drive/MyDrive/ANKA/"
    "run_02_english_5b/"
    "rmw3_full_5b_v3"
)

TRAIN_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

LATEST_PATH = (
    TRAIN_ROOT
    / "latest.pt"
)

FINAL_PATH = (
    TRAIN_ROOT
    / "final.pt"
)

RESULT_PATH = (
    TRAIN_ROOT
    / "result.json"
)


TARGET_TOKENS = 5_000_000_000

# Batch 96 caused OOM on the real corpus.
# Batch 64 safe VRAM with-headroom will-be-used.
BATCH_SIZE = 64

PEAK_LR = 2.5e-3
MIN_LR_RATIO = 0.10
WARMUP_RATIO = 0.02

WEIGHT_DECAY = 0.10
GRADIENT_CLIP = 1.0
AUXILIARY_WEIGHT = 0.01

LOG_EVERY = 10
VALIDATE_EVERY = 500
CHECKPOINT_EVERY = 250

# torch.compile fuses the element-wise work (RMSNorm, RoPE, SwiGLU,
# residual adds, cross-entropy) into fewer kernels. The math is
# unchanged. The released 5B run used eager mode (False).
USE_TORCH_COMPILE = True


TOKENS_PER_FULL_STEP = (
    BATCH_SIZE
    * SEQ_LEN
)

TOTAL_STEPS = math.ceil(
    TARGET_TOKENS
    / TOKENS_PER_FULL_STEP
)

WARMUP_STEPS = max(
    1,
    int(
        TOTAL_STEPS
        * WARMUP_RATIO
    ),
)


# ============================================================
# 4. OPTIMIZER
# ============================================================

def make_optimizer(
    training_model: torch.nn.Module,
):
    optimizer_arguments = {
        "lr": PEAK_LR,
        "betas": (0.9, 0.95),
        "eps": 1e-8,
        "weight_decay": WEIGHT_DECAY,
    }

    if (
        "fused"
        in inspect.signature(
            torch.optim.AdamW
        ).parameters
    ):
        optimizer_arguments[
            "fused"
        ] = True

    return torch.optim.AdamW(
        training_model.parameters(),
        **optimizer_arguments,
    )


# ============================================================
# 5. LOSS
# ============================================================

def calculate_loss(
    training_model: RMW3,
    input_tokens: torch.Tensor,
    target_tokens: torch.Tensor,
):
    logits, router_balance_loss = (
        training_model(
            input_tokens
        )
    )

    next_token_loss = F.cross_entropy(
        logits.reshape(
            -1,
            logits.shape[-1],
        ),
        target_tokens.reshape(-1),
        ignore_index=-100,
    )

    combined_loss = (
        next_token_loss
        + AUXILIARY_WEIGHT
        * router_balance_loss
    )

    return (
        combined_loss,
        next_token_loss.detach(),
        router_balance_loss.detach(),
    )


# ============================================================
# 6. LEARNING-RATE SCHEDULE
# ============================================================

def learning_rate(
    step: int,
) -> float:
    if step < WARMUP_STEPS:
        return (
            PEAK_LR
            * (step + 1)
            / WARMUP_STEPS
        )

    progress = (
        step - WARMUP_STEPS
    ) / max(
        1,
        TOTAL_STEPS
        - WARMUP_STEPS,
    )

    progress = min(
        max(
            progress,
            0.0,
        ),
        1.0,
    )

    cosine_value = (
        0.5
        * (
            1.0
            + math.cos(
                math.pi
                * progress
            )
        )
    )

    return (
        PEAK_LR
        * (
            MIN_LR_RATIO
            + (
                1.0
                - MIN_LR_RATIO
            )
            * cosine_value
        )
    )


# ============================================================
# 7. VALIDATION
# ============================================================

def evaluate_validation(
    validation_model: RMW3,
    maximum_batches: int = 16,
    validation_batch_size: int = 8,
) -> float:
    validation_model.eval()

    validation_losses = []

    validation_rows = (
        (len(VALIDATION_RAM) - 1)
        // SEQ_LEN
    )

    number_of_batches = min(
        maximum_batches,
        math.ceil(
            validation_rows
            / validation_batch_size
        ),
    )

    with torch.no_grad():
        for batch_index in range(
            number_of_batches
        ):
            first_row = (
                batch_index
                * validation_batch_size
            )

            final_row = min(
                first_row
                + validation_batch_size,
                validation_rows,
            )

            if first_row >= final_row:
                break

            packed = np.stack(
                [
                    VALIDATION_RAM[
                        row_index
                        * SEQ_LEN:
                        row_index
                        * SEQ_LEN
                        + SEQ_LEN
                        + 1
                    ]
                    for row_index in range(
                        first_row,
                        final_row,
                    )
                ]
            ).astype(
                np.int64,
                copy=False,
            )

            validation_inputs = (
                torch
                .from_numpy(
                    packed[:, :-1]
                )
                .to(
                    DEVICE,
                    non_blocking=True,
                )
            )

            validation_targets = (
                torch
                .from_numpy(
                    packed[:, 1:]
                )
                .to(
                    DEVICE,
                    non_blocking=True,
                )
            )

            with torch.autocast(
                device_type="cuda",
                dtype=torch.bfloat16,
            ):
                (
                    _,
                    current_language_loss,
                    _,
                ) = calculate_loss(
                    validation_model,
                    validation_inputs,
                    validation_targets,
                )

            validation_losses.append(
                float(
                    current_language_loss
                )
            )

            del (
                validation_inputs,
                validation_targets,
                packed,
            )

    validation_model.train()

    return float(
        np.mean(
            validation_losses
        )
    )


# ============================================================
# 8. ATOMIC CHECKPOINT
# ============================================================

def save_checkpoint_atomic(
    checkpoint_payload: dict,
    destination: Path,
) -> None:
    local_temporary = (
        Path("/content")
        / (
            destination.name
            + ".writing"
        )
    )

    drive_temporary = (
        destination.with_suffix(
            destination.suffix
            + ".writing"
        )
    )

    torch.save(
        checkpoint_payload,
        local_temporary,
    )

    shutil.copyfile(
        local_temporary,
        drive_temporary,
    )

    os.replace(
        drive_temporary,
        destination,
    )

    local_temporary.unlink(
        missing_ok=True
    )


def make_checkpoint_payload(
    checkpoint_model: RMW3,
    checkpoint_optimizer,
    completed_step: int,
    seen_tokens: int,
    best_validation_loss: float,
) -> dict:
    return {
        "version": 3,
        "architecture": "RMW3",
        "configuration": vars(
            RMW3_CONFIG
        ),
        "parameters": (
            RMW3_PARAMETERS
        ),
        "batch_size": (
            BATCH_SIZE
        ),
        "sequence_length": (
            SEQ_LEN
        ),
        "step": int(
            completed_step
        ),
        "seen_tokens": int(
            seen_tokens
        ),
        "best_validation_loss": float(
            best_validation_loss
        ),
        "model": (
            checkpoint_model
            .state_dict()
        ),
        "optimizer": (
            checkpoint_optimizer
            .state_dict()
        ),
        "python_rng": (
            random.getstate()
        ),
        "numpy_rng": (
            np.random.get_state()
        ),
        "torch_rng": (
            torch.get_rng_state()
        ),
        "cuda_rng": (
            torch.cuda
            .get_rng_state_all()
        ),
    }


# ============================================================
# 9. GPU MEMORY
# ============================================================

free_vram_bytes, total_vram_bytes = (
    torch.cuda.mem_get_info()
)

print(
    "GPU temizlendi."
)

print(
    "Free VRAM:",
    f"{free_vram_bytes / 2**30:.2f} GiB",
)

print(
    "Total VRAM:",
    f"{total_vram_bytes / 2**30:.2f} GiB",
)


# ============================================================
# 10. MODEL and OPTIMIZER
# ============================================================

model = RMW3(
    RMW3_CONFIG
).to(
    device=DEVICE,
    dtype=torch.bfloat16,
)

optimizer = make_optimizer(
    model
)


# Only the forward pass and loss are compiled. The optimizer,
# gradient clipping, validation and checkpoints keep using the
# uncompiled `model`, so checkpoint keys are unchanged.
def training_loss(
    input_tokens: torch.Tensor,
    target_tokens: torch.Tensor,
):
    return calculate_loss(
        model,
        input_tokens,
        target_tokens,
    )


compiled_training_loss = (
    torch.compile(training_loss)
    if USE_TORCH_COMPILE
    else training_loss
)


start_step = 0
completed_step = 0

seen_tokens = 0
initial_seen_tokens = 0

best_validation_loss = float(
    "inf"
)


# ============================================================
# 11. CHECKPOINT'TEN resume
# ============================================================

if LATEST_PATH.exists():
    print(
        "Checkpoint loading:",
        LATEST_PATH,
    )

    checkpoint = torch.load(
        LATEST_PATH,
        map_location="cpu",
        weights_only=False,
    )

    if (
        checkpoint.get("version")
        != 3
    ):
        raise RuntimeError(
            "Checkpoint surumu uyumsuz."
        )

    if (
        checkpoint.get(
            "architecture"
        )
        != "RMW3"
    ):
        raise RuntimeError(
            "Checkpoint mimarisi uyumsuz."
        )

    if (
        checkpoint.get(
            "configuration"
        )
        != vars(RMW3_CONFIG)
    ):
        raise RuntimeError(
            "Checkpoint model configi "
            "uyumsuz."
        )

    if (
        int(
            checkpoint.get(
                "parameters",
                -1,
            )
        )
        != RMW3_PARAMETERS
    ):
        raise RuntimeError(
            "Checkpoint parameter count "
            "uyumsuz."
        )

    model.load_state_dict(
        checkpoint["model"],
        strict=True,
    )

    optimizer.load_state_dict(
        checkpoint["optimizer"]
    )

    start_step = int(
        checkpoint["step"]
    )

    completed_step = (
        start_step
    )

    seen_tokens = int(
        checkpoint["seen_tokens"]
    )

    initial_seen_tokens = (
        seen_tokens
    )

    best_validation_loss = float(
        checkpoint.get(
            "best_validation_loss",
            float("inf"),
        )
    )

    random.setstate(
        checkpoint[
            "python_rng"
        ]
    )

    np.random.set_state(
        checkpoint[
            "numpy_rng"
        ]
    )

    torch.set_rng_state(
        checkpoint[
            "torch_rng"
        ]
    )

    torch.cuda.set_rng_state_all(
        checkpoint[
            "cuda_rng"
        ]
    )

    del checkpoint
    gc.collect()

    print(
        f"resume point: "
        f"step={start_step:,}, "
        f"token={seen_tokens:,}"
    )


# ============================================================
# 12. RAM LOADER
# ============================================================

batcher = ExactRamBatcher(
    NATURAL_RAM,
    WORLD_RAM,
    seed=DATA_SEED,
)

batcher.seek_tokens(
    seen_tokens
)

loader = PrefetchRamLoader(
    batcher=batcher,
    batch_size=BATCH_SIZE,
    target_tokens=TARGET_TOKENS,
    start_tokens=seen_tokens,
)


# ============================================================
# 13. TRAINING INFORMATION
# ============================================================

print("\n" + "=" * 88)
print("RMW3 FULL 5B TRAINING — SAFE BATCH")
print("=" * 88)

print(
    "GPU             :",
    torch.cuda.get_device_name(0),
)

print(
    "parameter       :",
    f"{RMW3_PARAMETERS:,}",
)

print(
    "Batch           :",
    BATCH_SIZE,
)

print(
    "Sequence        :",
    SEQ_LEN,
)

print(
    "Token/step      :",
    f"{TOKENS_PER_FULL_STEP:,}",
)

print(
    "Step            :",
    f"{start_step:,}/"
    f"{TOTAL_STEPS:,}",
)

print(
    "Token           :",
    f"{seen_tokens:,}/"
    f"{TARGET_TOKENS:,}",
)

print(
    "Checkpoint      :",
    LATEST_PATH,
)

print(
    "torch.compile   :",
    "on (first step compiles)"
    if USE_TORCH_COMPILE
    else "off",
)


# ============================================================
# 14. training
# ============================================================

model.train()

training_started = (
    time.perf_counter()
)

log_started = (
    training_started
)

log_start_tokens = (
    seen_tokens
)

last_language_loss = float(
    "nan"
)


try:
    for step, (
        cpu_inputs,
        cpu_labels,
        active_tokens,
        world_ratio,
    ) in enumerate(
        loader,
        start=start_step,
    ):
        current_learning_rate = (
            learning_rate(step)
        )

        for parameter_group in (
            optimizer.param_groups
        ):
            parameter_group[
                "lr"
            ] = current_learning_rate

        inputs = cpu_inputs.to(
            DEVICE,
            non_blocking=True,
        )

        labels = cpu_labels.to(
            DEVICE,
            non_blocking=True,
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.bfloat16,
        ):
            (
                total_loss,
                language_loss,
                balance_loss,
            ) = compiled_training_loss(
                inputs,
                labels,
            )

        if not torch.isfinite(
            total_loss
        ):
            raise FloatingPointError(
                "Loss NaN or Inf oldu."
            )

        total_loss.backward()

        gradient_norm = (
            torch.nn.utils
            .clip_grad_norm_(
                model.parameters(),
                GRADIENT_CLIP,
            )
        )

        if not torch.isfinite(
            gradient_norm
        ):
            raise FloatingPointError(
                "Gradient norm NaN or Inf oldu."
            )

        optimizer.step()

        seen_tokens += int(
            active_tokens
        )

        completed_step = (
            step + 1
        )

        last_language_loss = float(
            language_loss
        )

        del (
            inputs,
            labels,
            cpu_inputs,
            cpu_labels,
            total_loss,
        )

        if (
            completed_step
            % LOG_EVERY == 0
            or
            seen_tokens
            >= TARGET_TOKENS
        ):
            torch.cuda.synchronize()

            now = time.perf_counter()

            tokens_per_second = (
                seen_tokens
                - log_start_tokens
            ) / max(
                now - log_started,
                1e-6,
            )

            peak_vram_gib = (
                torch.cuda
                .max_memory_reserved()
                / 2**30
            )

            print(
                f"step="
                f"{completed_step:06d}/"
                f"{TOTAL_STEPS:06d} "
                f"loss="
                f"{last_language_loss:.4f} "
                f"aux="
                f"{float(balance_loss):.4f} "
                f"lr="
                f"{current_learning_rate:.3e} "
                f"grad="
                f"{float(gradient_norm):.3f} "
                f"tok/s="
                f"{tokens_per_second:,.0f} "
                f"seen="
                f"{seen_tokens:,} "
                f"world="
                f"{world_ratio * 100:.2f}% "
                f"peak="
                f"{peak_vram_gib:.2f}GiB"
            )

            log_started = now
            log_start_tokens = (
                seen_tokens
            )

        if (
            completed_step
            % VALIDATE_EVERY == 0
            or
            seen_tokens
            >= TARGET_TOKENS
        ):
            current_validation_loss = (
                evaluate_validation(
                    model
                )
            )

            best_validation_loss = min(
                best_validation_loss,
                current_validation_loss,
            )

            print(
                "validation_loss="
                f"{current_validation_loss:.6f} "
                "| best="
                f"{best_validation_loss:.6f}"
            )

        if (
            completed_step
            % CHECKPOINT_EVERY == 0
            or
            seen_tokens
            >= TARGET_TOKENS
        ):
            save_checkpoint_atomic(
                make_checkpoint_payload(
                    model,
                    optimizer,
                    completed_step,
                    seen_tokens,
                    best_validation_loss,
                ),
                LATEST_PATH,
            )

            print(
                "Checkpoint saved:",
                LATEST_PATH,
            )

        if (
            seen_tokens
            >= TARGET_TOKENS
        ):
            break


except BaseException:
    gc.collect()
    torch.cuda.empty_cache()

    save_checkpoint_atomic(
        make_checkpoint_payload(
            model,
            optimizer,
            completed_step,
            seen_tokens,
            best_validation_loss,
        ),
        LATEST_PATH,
    )

    print(
        "interruption checkpointi saved:",
        LATEST_PATH,
    )

    raise


finally:
    loader.close()


# ============================================================
# 15. FINAL SAVE
# ============================================================

training_seconds = (
    time.perf_counter()
    - training_started
)

save_checkpoint_atomic(
    make_checkpoint_payload(
        model,
        optimizer,
        completed_step,
        seen_tokens,
        best_validation_loss,
    ),
    FINAL_PATH,
)


result = {
    "arm": "RMW3",
    "parameters": (
        RMW3_PARAMETERS
    ),
    "seen_tokens": (
        seen_tokens
    ),
    "target_tokens": (
        TARGET_TOKENS
    ),
    "batch_size": (
        BATCH_SIZE
    ),
    "sequence_length": (
        SEQ_LEN
    ),
    "completed_steps": (
        completed_step
    ),
    "last_training_loss": (
        last_language_loss
    ),
    "best_validation_loss": (
        best_validation_loss
    ),
    "wall_seconds": (
        training_seconds
    ),
    "average_tokens_per_second": (
        seen_tokens
        - initial_seen_tokens
    ) / max(
        training_seconds,
        1e-6,
    ),
    "latest_checkpoint": str(
        LATEST_PATH
    ),
    "final_checkpoint": str(
        FINAL_PATH
    ),
}


RESULT_PATH.write_text(
    json.dumps(
        result,
        indent=2,
    ),
    encoding="utf-8",
)


print("\n" + "=" * 88)
print("TRAINING COMPLETED")
print("=" * 88)

print(
    json.dumps(
        result,
        indent=2,
    )
)

## 5. Configure the official evaluation environment

This section evaluates an existing checkpoint and saves machine-readable results. It should not change the released model weights.


In [ ]:
# ============================================================
# cell 1 — SETUP AND SETTINGS
# Run after training completes.
# Do not restart the runtime.
# ============================================================

%pip install -q --upgrade "lm_eval==0.4.12" "datasets>=3.0" "tokenizers>=0.20"

from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version
from types import SimpleNamespace

import csv
import gc
import json
import math
import os
import random
import time

import numpy as np
import torch
import torch.nn.functional as F

from tokenizers import Tokenizer
from tqdm.auto import tqdm


SEED = 3407

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True


RUN_ROOT = Path(
    "/content/drive/MyDrive/ANKA/run_02_english_5b"
)

TRAIN_ROOT = (
    RUN_ROOT / "rmw3_full_5b_v3"
)

FINAL_CHECKPOINT = (
    TRAIN_ROOT / "final.pt"
)

TOKENIZER_PATH = (
    RUN_ROOT
    / "tokenizer"
    / "tokenizer.json"
)

REPORT_ROOT = (
    RUN_ROOT
    / "official_benchmark_v2"
)

TASK_ROOT = (
    REPORT_ROOT / "tasks"
)

REPORT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

TASK_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["HF_DATASETS_CACHE"] = (
    "/content/hf_cache/datasets"
)
os.environ["TOKENIZERS_PARALLELISM"] = "false"


assert torch.cuda.is_available(), (
    "CUDA GPU not found."
)

assert FINAL_CHECKPOINT.exists(), (
    f"Final checkpoint not found: "
    f"{FINAL_CHECKPOINT}"
)

assert TOKENIZER_PATH.exists(), (
    f"Tokenizer not found: "
    f"{TOKENIZER_PATH}"
)

assert "RMW3" in globals(), (
    "RMW3 class is not loaded in memory. "
    "only RMW3 model definition cell run."
)

assert "RMW3_CONFIG" in globals(), (
    "RMW3_CONFIG is not loaded in memory. "
    "only RMW3 model definition cell run."
)


print("GPU        :", torch.cuda.get_device_name(0))
print("PyTorch    :", torch.__version__)
print("lm_eval    :", version("lm_eval"))
print("Checkpoint :", FINAL_CHECKPOINT)
print("report path :", REPORT_ROOT)

## 6. Load and verify the final checkpoint

This is a compute-intensive stage. Confirm the selected runtime, input paths, token budget, and resume directory before execution. Checkpoints are written incrementally.


In [ ]:
# ============================================================
# cell 2 — LOAD FINAL CHECKPOINT
# ============================================================

# Stop the training DataLoader.
for loader_name in (
    "loader",
    "train_loader",
    "prefetch_loader",
):
    loader_object = globals().get(loader_name)

    if hasattr(loader_object, "close"):
        try:
            loader_object.close()
        except Exception:
            pass


# Release large RAM/GPU objects left by training.
objects_to_clear = (
    "model",
    "optimizer",
    "scheduler",
    "loader",
    "train_loader",
    "prefetch_loader",
    "inputs",
    "targets",
    "logits",
    "loss",
    "language_loss",
    "balance_loss",
    "NATURAL_RAM",
    "WORLD_RAM",
    "VALIDATION_RAM",
    "benchmark_model",
    "anka_lm",
)

for object_name in objects_to_clear:
    if object_name in globals():
        globals()[object_name] = None

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()
torch.cuda.synchronize()

free_vram, total_vram = torch.cuda.mem_get_info()

print(
    "Free VRAM after cleanup:",
    f"{free_vram / 2**30:.2f} GiB",
)


checkpoint = torch.load(
    FINAL_CHECKPOINT,
    map_location="cpu",
    weights_only=False,
)


def read_checkpoint_integer(
    payload: dict,
    possible_names: tuple[str, ...],
    default: int = 0,
) -> int:
    for name in possible_names:
        if payload.get(name) is not None:
            return int(payload[name])

    return int(default)


SEEN_TOKENS = read_checkpoint_integer(
    checkpoint,
    (
        "seen_tokens",
        "tokens_seen",
        "processed_tokens",
    ),
)

COMPLETED_STEPS = read_checkpoint_integer(
    checkpoint,
    (
        "completed_steps",
        "global_step",
        "step",
    ),
)

TARGET_TOKENS = read_checkpoint_integer(
    checkpoint,
    (
        "target_tokens",
        "token_budget",
    ),
    default=5_000_000_000,
)

assert SEEN_TOKENS == 5_000_000_000, (
    f"Final checkpoint is not the 5B run: "
    f"{SEEN_TOKENS:,}"
)

assert TARGET_TOKENS == 5_000_000_000

assert COMPLETED_STEPS == 76_294, (
    f"Unexpected step: "
    f"{COMPLETED_STEPS:,}"
)


def extract_state_dict(
    payload: dict,
) -> dict[str, torch.Tensor]:
    for key in (
        "model",
        "model_state_dict",
        "state_dict",
        "model_weights",
    ):
        candidate = payload.get(key)

        if (
            isinstance(candidate, dict)
            and candidate
            and all(
                torch.is_tensor(value)
                for value in candidate.values()
            )
        ):
            return candidate

    if (
        payload
        and all(
            torch.is_tensor(value)
            for value in payload.values()
        )
    ):
        return payload

    raise RuntimeError(
        "state_dict not found in checkpoint."
    )


def remove_uniform_prefix(
    state_dict: dict[str, torch.Tensor],
    prefix: str,
) -> dict[str, torch.Tensor]:
    if state_dict and all(
        name.startswith(prefix)
        for name in state_dict
    ):
        return {
            name[len(prefix):]: tensor
            for name, tensor in state_dict.items()
        }

    return state_dict


state_dict = extract_state_dict(
    checkpoint
)

for possible_prefix in (
    "_orig_mod.",
    "module.",
    "model.",
):
    state_dict = remove_uniform_prefix(
        state_dict,
        possible_prefix,
    )


benchmark_model = RMW3(
    RMW3_CONFIG
)

load_result = benchmark_model.load_state_dict(
    state_dict,
    strict=True,
)

assert not load_result.missing_keys, (
    load_result.missing_keys
)

assert not load_result.unexpected_keys, (
    load_result.unexpected_keys
)

benchmark_model = benchmark_model.to(
    device="cuda",
    dtype=torch.bfloat16,
)

benchmark_model.eval()


PARAMETER_COUNT = sum(
    parameter.numel()
    for parameter in benchmark_model.parameters()
)

assert PARAMETER_COUNT == 48_944_657, (
    f"Incorrect parameter count: "
    f"{PARAMETER_COUNT:,}"
)

assert PARAMETER_COUNT <= 50_000_000


tokenizer = Tokenizer.from_file(
    str(TOKENIZER_PATH)
)

VOCAB_SIZE = tokenizer.get_vocab_size(
    with_added_tokens=True
)

PAD_ID = tokenizer.token_to_id("<pad>")
BOS_ID = tokenizer.token_to_id("<empty>")
EOS_ID = tokenizer.token_to_id("<eos>")
UNK_ID = tokenizer.token_to_id("<unk>")

assert VOCAB_SIZE == 16_384
assert PAD_ID == 0
assert BOS_ID == 1
assert EOS_ID == 2
assert UNK_ID == 3


def extract_model_logits(
    output,
) -> torch.Tensor:
    if torch.is_tensor(output):
        return output

    if isinstance(output, (tuple, list)):
        if not output:
            raise RuntimeError(
                "Model returned an empty tuple/list."
            )

        return output[0]

    if isinstance(output, dict):
        return output["logits"]

    if hasattr(output, "logits"):
        return output.logits

    raise TypeError(
        f"Could not obtain logits: {type(output)}"
    )


probe = torch.tensor(
    [
        [
            BOS_ID,
            tokenizer.token_to_id("the")
            if tokenizer.token_to_id("the") is not None
            else EOS_ID,
            EOS_ID,
        ]
    ],
    device="cuda",
    dtype=torch.long,
)

with torch.inference_mode():
    with torch.autocast(
        device_type="cuda",
        dtype=torch.bfloat16,
    ):
        probe_output = benchmark_model(probe)

    probe_logits = extract_model_logits(
        probe_output
    )

assert probe_logits.shape == (
    1,
    3,
    VOCAB_SIZE,
)

assert torch.isfinite(
    probe_logits.float()
).all()

TRAINING_METADATA = {
    "arm": checkpoint.get("arm", "RMW3"),
    "seen_tokens": SEEN_TOKENS,
    "target_tokens": TARGET_TOKENS,
    "completed_steps": COMPLETED_STEPS,
    "wall_seconds": checkpoint.get(
        "wall_seconds",
        42_409.015676945004,
    ),
    "average_tokens_per_second": checkpoint.get(
        "average_tokens_per_second",
        117_899.45888129094,
    ),
    "best_validation_loss": checkpoint.get(
        "best_validation_loss",
        2.3343789130449295,
    ),
}

del probe
del probe_output
del probe_logits
del state_dict
del checkpoint

gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

print("Final checkpoint     : PASS")
print("parameter            :", f"{PARAMETER_COUNT:,}")
print("Token                :", f"{SEEN_TOKENS:,}")
print("Step                 :", f"{COMPLETED_STEPS:,}")
print("Validation loss      :", TRAINING_METADATA["best_validation_loss"])

## 7. Build and mathematically verify the lm-evaluation-harness adapter

This section evaluates an existing checkpoint and saves machine-readable results. It should not change the released model weights.


In [ ]:
# ============================================================
# cell 3 — LM-EVAL ADAPTER AND MATHEMATICAL VERIFICATION
# ============================================================

from lm_eval.api.model import TemplateLM
from lm_eval import utils as lm_eval_utils


class ANKAHarnessLM(TemplateLM):
    """RMW3 adapts the model to lm-evaluation-harness."""

    def __init__(
        self,
        model,
        tokenizer,
        batch_size: int = 16,
        max_length: int = 1024,
        max_gen_toks: int = 128,
    ):
        super().__init__()

        self.model = model
        self.tokenizer = tokenizer

        self._batch_size = int(batch_size)
        self._max_length = int(max_length)
        self._max_gen_toks = int(max_gen_toks)

        self._device = torch.device("cuda")
        self._rank = 0
        self._world_size = 1

        self.pad_id = int(PAD_ID)
        self.bos_id = int(BOS_ID)
        self.eos_id = int(EOS_ID)
        self.vocab_size = int(VOCAB_SIZE)

    @property
    def eot_token_id(self) -> int:
        return self.eos_id

    @property
    def prefix_token_id(self) -> int:
        return self.bos_id

    @property
    def max_length(self) -> int:
        return self._max_length

    @property
    def max_gen_toks(self) -> int:
        return self._max_gen_toks

    @property
    def batch_size(self) -> int:
        return self._batch_size

    @property
    def device(self):
        return self._device

    @property
    def rank(self) -> int:
        return self._rank

    @property
    def world_size(self) -> int:
        return self._world_size

    @property
    def tokenizer_name(self) -> str:
        return "anka_english_16384_v1"

    def tok_encode(
        self,
        string: str,
        add_special_tokens=None,
        **kwargs,
    ) -> list[int]:
        if not string:
            return []

        return self.tokenizer.encode(
            string,
            add_special_tokens=False,
        ).ids

    def tok_decode(
        self,
        tokens: list[int],
    ) -> str:
        return self.tokenizer.decode(
            tokens,
            skip_special_tokens=True,
        )

    def _forward_logits(
        self,
        input_ids: torch.Tensor,
    ) -> torch.Tensor:
        with torch.inference_mode():
            with torch.autocast(
                device_type="cuda",
                dtype=torch.bfloat16,
            ):
                output = self.model(input_ids)

        logits = extract_model_logits(output)

        if logits.ndim != 3:
            raise RuntimeError(
                f"Incorrect logits rank: "
                f"{tuple(logits.shape)}"
            )

        if logits.shape[-1] != self.vocab_size:
            raise RuntimeError(
                f"Vocabulary mismatch: "
                f"{logits.shape[-1]} != "
                f"{self.vocab_size}"
            )

        return logits

    def _loglikelihood_tokens(
        self,
        requests,
        disable_tqdm: bool = False,
        **kwargs,
    ):
        results = [None] * len(requests)
        prepared = []

        for request_index, request in enumerate(
            requests
        ):
            _, context_tokens, continuation_tokens = (
                request
            )

            context_tokens = list(context_tokens)
            continuation_tokens = list(
                continuation_tokens
            )

            if not continuation_tokens:
                results[request_index] = (
                    0.0,
                    True,
                )
                continue

            if not context_tokens:
                context_tokens = [
                    self.prefix_token_id
                ]

            if (
                len(continuation_tokens)
                > self.max_length
            ):
                raise ValueError(
                    "Continuation exceeds the 1,024-token context window."
                )

            maximum_context = (
                self.max_length
                + 1
                - len(continuation_tokens)
            )

            kept_context = context_tokens[
                -maximum_context:
            ]

            if not kept_context:
                kept_context = [
                    self.prefix_token_id
                ]

            sequence = (
                kept_context
                + continuation_tokens
            )

            input_tokens = sequence[:-1]

            continuation_start = (
                len(kept_context) - 1
            )

            if len(input_tokens) > self.max_length:
                raise RuntimeError(
                    "Model context limit exceeded."
                )

            prepared.append(
                {
                    "request_index": request_index,
                    "input_tokens": input_tokens,
                    "target_tokens": continuation_tokens,
                    "continuation_start": (
                        continuation_start
                    ),
                }
            )

        prepared.sort(
            key=lambda item: len(
                item["input_tokens"]
            ),
            reverse=True,
        )

        batch_starts = range(
            0,
            len(prepared),
            self.batch_size,
        )

        if not disable_tqdm:
            batch_starts = tqdm(
                batch_starts,
                total=math.ceil(
                    len(prepared)
                    / self.batch_size
                ) if prepared else 0,
                desc="ANKA likelihood",
            )

        for batch_start in batch_starts:
            batch_items = prepared[
                batch_start:
                batch_start + self.batch_size
            ]

            maximum_input_length = max(
                len(item["input_tokens"])
                for item in batch_items
            )

            input_batch = torch.full(
                (
                    len(batch_items),
                    maximum_input_length,
                ),
                self.pad_id,
                device=self.device,
                dtype=torch.long,
            )

            for row_index, item in enumerate(
                batch_items
            ):
                item_length = len(
                    item["input_tokens"]
                )

                input_batch[
                    row_index,
                    :item_length,
                ] = torch.tensor(
                    item["input_tokens"],
                    device=self.device,
                    dtype=torch.long,
                )

            logits = self._forward_logits(
                input_batch
            )

            for row_index, item in enumerate(
                batch_items
            ):
                start = item[
                    "continuation_start"
                ]

                target_ids = item[
                    "target_tokens"
                ]

                end = start + len(target_ids)

                selected_logits = logits[
                    row_index,
                    start:end,
                    :,
                ].float()

                target_tensor = torch.tensor(
                    target_ids,
                    device=self.device,
                    dtype=torch.long,
                )

                if (
                    selected_logits.shape[0]
                    != target_tensor.shape[0]
                ):
                    raise RuntimeError(
                        "Logits/target alignment error."
                    )

                log_probabilities = (
                    F.log_softmax(
                        selected_logits,
                        dim=-1,
                    )
                    .gather(
                        dim=-1,
                        index=target_tensor.unsqueeze(-1),
                    )
                    .squeeze(-1)
                )

                log_probability = float(
                    log_probabilities.sum().item()
                )

                greedy_match = bool(
                    selected_logits.argmax(dim=-1)
                    .eq(target_tensor)
                    .all()
                    .item()
                )

                results[
                    item["request_index"]
                ] = (
                    log_probability,
                    greedy_match,
                )

            del input_batch
            del logits

        if not all(
            result is not None
            for result in results
        ):
            raise RuntimeError(
                "A likelihood result is missing."
            )

        return results

    def loglikelihood_rolling(
        self,
        requests,
        disable_tqdm: bool = False,
    ) -> list[float]:
        results = []

        iterator = requests

        if not disable_tqdm:
            iterator = tqdm(
                requests,
                desc="ANKA rolling likelihood",
            )

        for request in iterator:
            text = request.args[0]
            token_ids = self.tok_encode(text)

            if not token_ids:
                results.append(0.0)
                continue

            windows = list(
                map(
                    lm_eval_utils.make_disjoint_window,
                    lm_eval_utils.get_rolling_token_windows(
                        token_list=token_ids,
                        prefix_token=(
                            self.prefix_token_id
                        ),
                        max_seq_len=(
                            self.max_length - 1
                        ),
                        context_len=1,
                    ),
                )
            )

            scoring_requests = [
                (
                    None,
                    context_tokens,
                    continuation_tokens,
                )
                for (
                    context_tokens,
                    continuation_tokens,
                ) in windows
            ]

            window_results = (
                self._loglikelihood_tokens(
                    scoring_requests,
                    disable_tqdm=True,
                )
            )

            total_log_probability = float(
                sum(
                    score[0]
                    for score in window_results
                )
            )

            results.append(
                total_log_probability
            )

            if hasattr(self, "cache_hook"):
                self.cache_hook.add_partial(
                    "loglikelihood_rolling",
                    (text,),
                    total_log_probability,
                )

        return results

    def _generate_one(
        self,
        prompt: str,
        generation_kwargs: dict,
    ) -> str:
        maximum_new_tokens = min(
            int(
                generation_kwargs.get(
                    "max_gen_toks",
                    self.max_gen_toks,
                )
            ),
            self.max_gen_toks,
        )

        stop_strings = generation_kwargs.get(
            "until",
            [],
        )

        if isinstance(stop_strings, str):
            stop_strings = [stop_strings]

        stop_strings = [
            value
            for value in stop_strings
            if isinstance(value, str)
            and value
        ]

        temperature = float(
            generation_kwargs.get(
                "temperature",
                0.0,
            )
        )

        top_p = float(
            generation_kwargs.get(
                "top_p",
                1.0,
            )
        )

        do_sample = bool(
            generation_kwargs.get(
                "do_sample",
                temperature > 0,
            )
        )

        prompt_ids = self.tok_encode(prompt)

        if not prompt_ids:
            prompt_ids = [
                self.prefix_token_id
            ]

        generated_ids = []

        for _ in range(maximum_new_tokens):
            active_ids = (
                prompt_ids + generated_ids
            )[-self.max_length:]

            input_tensor = torch.tensor(
                [active_ids],
                device=self.device,
                dtype=torch.long,
            )

            next_logits = self._forward_logits(
                input_tensor
            )[0, -1].float()

            if do_sample:
                safe_temperature = max(
                    temperature,
                    1e-5,
                )

                probabilities = torch.softmax(
                    next_logits / safe_temperature,
                    dim=-1,
                )

                if top_p < 1.0:
                    (
                        sorted_probabilities,
                        sorted_indices,
                    ) = torch.sort(
                        probabilities,
                        descending=True,
                    )

                    cumulative = torch.cumsum(
                        sorted_probabilities,
                        dim=-1,
                    )

                    remove = (
                        cumulative
                        - sorted_probabilities
                    ) > top_p

                    sorted_probabilities[
                        remove
                    ] = 0.0

                    denominator = (
                        sorted_probabilities.sum()
                    )

                    if denominator <= 0:
                        next_id = int(
                            next_logits.argmax().item()
                        )
                    else:
                        sorted_probabilities /= (
                            denominator
                        )

                        sampled_position = (
                            torch.multinomial(
                                sorted_probabilities,
                                num_samples=1,
                            )
                        )

                        next_id = int(
                            sorted_indices[
                                sampled_position
                            ].item()
                        )
                else:
                    next_id = int(
                        torch.multinomial(
                            probabilities,
                            num_samples=1,
                        ).item()
                    )
            else:
                next_id = int(
                    next_logits.argmax().item()
                )

            del input_tensor
            del next_logits

            if next_id == self.eos_id:
                break

            generated_ids.append(next_id)

            decoded = self.tok_decode(
                generated_ids
            )

            found_stops = [
                decoded.find(stop)
                for stop in stop_strings
                if stop in decoded
            ]

            if found_stops:
                return decoded[
                    :min(found_stops)
                ]

        return self.tok_decode(
            generated_ids
        )

    def generate_until(
        self,
        requests,
        disable_tqdm: bool = False,
    ) -> list[str]:
        outputs = []

        iterator = requests

        if not disable_tqdm:
            iterator = tqdm(
                requests,
                desc="ANKA generation",
            )

        for request in iterator:
            prompt, generation_kwargs = (
                request.args
            )

            outputs.append(
                self._generate_one(
                    prompt,
                    dict(generation_kwargs),
                )
            )

        return outputs


EVAL_BATCH_SIZE = 16

anka_lm = ANKAHarnessLM(
    model=benchmark_model,
    tokenizer=tokenizer,
    batch_size=EVAL_BATCH_SIZE,
    max_length=1024,
    max_gen_toks=128,
)


# ------------------------------------------------------------
# Mathematical score verification
# ------------------------------------------------------------

class TestRequest:
    def __init__(self, *args):
        self.args = args


test_context = "The capital of France is"
test_continuation = " Paris"

harness_result = anka_lm.loglikelihood(
    [
        TestRequest(
            test_context,
            test_continuation,
        )
    ],
    disable_tqdm=True,
)[0][0]


whole_ids = tokenizer.encode(
    test_context + test_continuation,
    add_special_tokens=False,
).ids

context_ids = tokenizer.encode(
    test_context,
    add_special_tokens=False,
).ids

continuation_ids = whole_ids[
    len(context_ids):
]

manual_input = torch.tensor(
    [whole_ids[:-1]],
    device="cuda",
    dtype=torch.long,
)

with torch.inference_mode():
    manual_logits = anka_lm._forward_logits(
        manual_input
    )

manual_selected_logits = manual_logits[
    0,
    len(context_ids) - 1:
    len(context_ids) - 1
    + len(continuation_ids),
    :,
].float()

manual_targets = torch.tensor(
    continuation_ids,
    device="cuda",
    dtype=torch.long,
)

manual_result = float(
    F.log_softmax(
        manual_selected_logits,
        dim=-1,
    )
    .gather(
        -1,
        manual_targets.unsqueeze(-1),
    )
    .sum()
    .item()
)

score_difference = abs(
    harness_result - manual_result
)

assert score_difference < 1e-4, (
    f"Likelihood verification failed: "
    f"{score_difference}"
)


rolling_result = (
    anka_lm.loglikelihood_rolling(
        [
            TestRequest(
                "Science helps people understand "
                "the natural world."
            )
        ],
        disable_tqdm=True,
    )
)

assert len(rolling_result) == 1
assert math.isfinite(rolling_result[0])

del manual_input
del manual_logits
del manual_selected_logits
del manual_targets

gc.collect()
torch.cuda.empty_cache()

print("Likelihood testi         : PASS")
print("Rolling likelihood testi : PASS")
print("Score difference               :", score_difference)
print("Evaluation batch         :", EVAL_BATCH_SIZE)

## 8. Run the four-task smoke test

Defines or validates model components used by the experiment. Keep tensor shapes, parameter counts, and causal behavior unchanged when reproducing the run.


In [ ]:
# ============================================================
# cell 4 — FOUR-TASK SMOKE TEST
# ============================================================

import lm_eval


OFFICIAL_TASKS = (
    "hellaswag",
    "arc_easy",
    "piqa",
    "winogrande",
)

NUM_FEWSHOT = 0
SMOKE_LIMIT = 32


def json_default(value):
    if isinstance(value, np.generic):
        return value.item()

    if torch.is_tensor(value):
        return value.detach().cpu().tolist()

    if isinstance(value, Path):
        return str(value)

    if isinstance(value, set):
        return sorted(value)

    return str(value)


def atomic_json_save(
    destination: Path,
    payload,
) -> None:
    temporary_path = destination.with_suffix(
        destination.suffix + ".tmp"
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            payload,
            file,
            ensure_ascii=False,
            indent=2,
            default=json_default,
        )

    temporary_path.replace(destination)


smoke_start = time.perf_counter()

smoke_results = lm_eval.simple_evaluate(
    model=anka_lm,
    tasks=list(OFFICIAL_TASKS),
    num_fewshot=NUM_FEWSHOT,
    batch_size=EVAL_BATCH_SIZE,
    device="cuda:0",
    limit=SMOKE_LIMIT,
    log_samples=True,
    random_seed=SEED,
    numpy_random_seed=SEED,
    torch_random_seed=SEED,
    fewshot_random_seed=SEED,
)

assert smoke_results is not None

for task_name in OFFICIAL_TASKS:
    assert task_name in smoke_results["results"]

    task_metrics = smoke_results[
        "results"
    ][task_name]

    numeric_metrics = [
        float(value)
        for value in task_metrics.values()
        if isinstance(
            value,
            (int, float, np.number),
        )
    ]

    assert numeric_metrics
    assert all(
        math.isfinite(value)
        for value in numeric_metrics
    )

smoke_seconds = (
    time.perf_counter() - smoke_start
)

atomic_json_save(
    REPORT_ROOT / "smoke_test.json",
    {
        "status": "PASS",
        "limit_per_task": SMOKE_LIMIT,
        "num_fewshot": NUM_FEWSHOT,
        "elapsed_seconds": smoke_seconds,
        "results": smoke_results,
    },
)

print("\n" + "=" * 80)
print("SMOKE TEST: PASS")
print("=" * 80)
print(f"Duration: {smoke_seconds / 60:.2f} minutes")

for task_name in OFFICIAL_TASKS:
    print(
        "\n",
        task_name,
        smoke_results["results"][task_name],
    )

## 9. Run the full official zero-shot benchmarks

This section evaluates an existing checkpoint and saves machine-readable results. It should not change the released model weights.


In [ ]:
# ============================================================
# cell 5 — FULL OFFICIAL BENCHMARKS
# Each completed task is saved to a separate Drive file.
# Completed tasks are skipped after interruption.
# ============================================================

FULL_TASK_RESULTS = {}

for task_name in OFFICIAL_TASKS:
    output_path = (
        TASK_ROOT
        / f"{task_name}_zero_shot.json"
    )

    print("\n" + "=" * 88)
    print(task_name.upper())
    print("=" * 88)

    if output_path.exists():
        try:
            with output_path.open(
                "r",
                encoding="utf-8",
            ) as file:
                existing_result = json.load(file)

            if existing_result.get("status") == "PASS":
                FULL_TASK_RESULTS[
                    task_name
                ] = existing_result

                print(
                    "Completed result found, "
                    "skipping:",
                    output_path,
                )
                continue
        except Exception:
            print(
                "Could not read old result; task "
                "again runilacak."
            )

    task_start = time.perf_counter()

    result = lm_eval.simple_evaluate(
        model=anka_lm,
        tasks=[task_name],
        num_fewshot=NUM_FEWSHOT,
        batch_size=EVAL_BATCH_SIZE,
        device="cuda:0",
        limit=None,
        log_samples=True,
        random_seed=SEED,
        numpy_random_seed=SEED,
        torch_random_seed=SEED,
        fewshot_random_seed=SEED,
    )

    if result is None:
        raise RuntimeError(
            f"{task_name} returned no result."
        )

    if task_name not in result["results"]:
        raise RuntimeError(
            f"{task_name} missing from the result dictionary."
        )

    task_seconds = (
        time.perf_counter() - task_start
    )

    payload = {
        "status": "PASS",
        "task": task_name,
        "num_fewshot": NUM_FEWSHOT,
        "elapsed_seconds": task_seconds,
        "checkpoint": str(FINAL_CHECKPOINT),
        "parameters": PARAMETER_COUNT,
        "seen_tokens": SEEN_TOKENS,
        "harness": result,
    }

    atomic_json_save(
        output_path,
        payload,
    )

    FULL_TASK_RESULTS[
        task_name
    ] = payload

    print(
        f"\n{task_name} completed."
    )
    print(
        f"Duration: {task_seconds / 60:.2f} minutes"
    )
    print(
        result["results"][task_name]
    )

    gc.collect()
    torch.cuda.empty_cache()

print("\nFOUR OFFICIAL TASKS COMPLETED: PASS")

## 10. Evaluate WikiText-103 perplexity

This section evaluates an existing checkpoint and saves machine-readable results. It should not change the released model weights.


In [ ]:
# ============================================================
# cell 6 — WIKITEXT-103 PERPLEXITY
# ============================================================

from datasets import load_dataset


WIKITEXT_OUTPUT = (
    REPORT_ROOT
    / "wikitext103_test_perplexity.json"
)

WIKI_BATCH_SIZE = 8
WIKI_STRIDE = 512
MAX_CONTEXT = 1024


def calculate_wikitext103() -> dict:
    print(
        "Salesforce/wikitext "
        "wikitext-103-raw-v1/test loading..."
    )

    dataset = load_dataset(
        "Salesforce/wikitext",
        "wikitext-103-raw-v1",
        split="test",
    )

    documents = [
        row["text"].strip()
        for row in dataset
        if row["text"].strip()
    ]

    assert documents, (
        "WikiText-103 test split is empty."
    )

    full_text = "\n\n".join(documents)

    text_token_ids = tokenizer.encode(
        full_text,
        add_special_tokens=False,
    ).ids

    assert len(text_token_ids) > 10_000

    stream = [
        BOS_ID,
        *text_token_ids,
    ]

    windows = []

    for target_begin in range(
        1,
        len(stream),
        WIKI_STRIDE,
    ):
        target_end = min(
            target_begin + WIKI_STRIDE,
            len(stream),
        )

        context_begin = max(
            0,
            target_end - (MAX_CONTEXT + 1),
        )

        sequence = stream[
            context_begin:target_end
        ]

        input_ids = sequence[:-1]
        target_ids = sequence[1:]

        score_start = max(
            0,
            target_begin
            - (context_begin + 1),
        )

        assert len(input_ids) <= MAX_CONTEXT
        assert 0 <= score_start < len(target_ids)

        windows.append(
            (
                input_ids,
                target_ids,
                score_start,
            )
        )

    total_nll = 0.0
    total_scored_tokens = 0

    torch.cuda.reset_peak_memory_stats()

    evaluation_start = time.perf_counter()

    for batch_start in tqdm(
        range(
            0,
            len(windows),
            WIKI_BATCH_SIZE,
        ),
        desc="WikiText-103",
    ):
        batch_windows = windows[
            batch_start:
            batch_start + WIKI_BATCH_SIZE
        ]

        longest_input = max(
            len(window[0])
            for window in batch_windows
        )

        input_batch = torch.full(
            (
                len(batch_windows),
                longest_input,
            ),
            PAD_ID,
            device="cuda",
            dtype=torch.long,
        )

        for row_index, window in enumerate(
            batch_windows
        ):
            input_ids = window[0]

            input_batch[
                row_index,
                :len(input_ids),
            ] = torch.tensor(
                input_ids,
                device="cuda",
                dtype=torch.long,
            )

        batch_logits = (
            anka_lm._forward_logits(
                input_batch
            )
        )

        for row_index, (
            _,
            target_ids,
            score_start,
        ) in enumerate(batch_windows):
            score_end = len(target_ids)

            selected_logits = batch_logits[
                row_index,
                score_start:score_end,
                :,
            ].float()

            selected_targets = torch.tensor(
                target_ids[
                    score_start:score_end
                ],
                device="cuda",
                dtype=torch.long,
            )

            if (
                selected_logits.shape[0]
                != selected_targets.shape[0]
            ):
                raise RuntimeError(
                    "WikiText logits/target "
                    "alignment error."
                )

            batch_nll = F.cross_entropy(
                selected_logits,
                selected_targets,
                reduction="sum",
            )

            total_nll += float(
                batch_nll.item()
            )

            total_scored_tokens += int(
                selected_targets.numel()
            )

        del input_batch
        del batch_logits

    evaluation_seconds = (
        time.perf_counter() - evaluation_start
    )

    assert (
        total_scored_tokens
        == len(text_token_ids)
    ), (
        f"Incorrect token count: "
        f"{total_scored_tokens:,} != "
        f"{len(text_token_ids):,}"
    )

    mean_loss = (
        total_nll
        / total_scored_tokens
    )

    perplexity = math.exp(mean_loss)

    result = {
        "status": "PASS",
        "dataset": "Salesforce/wikitext",
        "config": "wikitext-103-raw-v1",
        "split": "test",
        "documents": len(documents),
        "tokenizer_tokens": len(
            text_token_ids
        ),
        "scored_tokens": (
            total_scored_tokens
        ),
        "context_length": MAX_CONTEXT,
        "stride": WIKI_STRIDE,
        "mean_loss": mean_loss,
        "perplexity": perplexity,
        "elapsed_seconds": (
            evaluation_seconds
        ),
        "tokens_per_second": (
            total_scored_tokens
            / evaluation_seconds
        ),
        "peak_vram_gib": (
            torch.cuda.max_memory_allocated()
            / 2**30
        ),
    }

    return result


if WIKITEXT_OUTPUT.exists():
    with WIKITEXT_OUTPUT.open(
        "r",
        encoding="utf-8",
    ) as file:
        WIKITEXT_RESULT = json.load(file)

    if WIKITEXT_RESULT.get("status") != "PASS":
        WIKITEXT_RESULT = (
            calculate_wikitext103()
        )

        atomic_json_save(
            WIKITEXT_OUTPUT,
            WIKITEXT_RESULT,
        )
else:
    WIKITEXT_RESULT = (
        calculate_wikitext103()
    )

    atomic_json_save(
        WIKITEXT_OUTPUT,
        WIKITEXT_RESULT,
    )


print("\n" + "=" * 80)
print("WIKITEXT-103 RESULT")
print("=" * 80)
print(
    "Loss       :",
    f"{WIKITEXT_RESULT['mean_loss']:.6f}",
)
print(
    "Perplexity :",
    f"{WIKITEXT_RESULT['perplexity']:.3f}",
)
print(
    "Token      :",
    f"{WIKITEXT_RESULT['scored_tokens']:,}",
)
print(
    "Speed        :",
    f"{WIKITEXT_RESULT['tokens_per_second']:,.0f} token/s",
)

## 11. Generate samples and save the final evaluation summary

This section evaluates an existing checkpoint and saves machine-readable results. It should not change the released model weights.


In [ ]:
# ============================================================
# cell 7 — GENERATION + FINAL JSON/CSV REPORT
# ============================================================

GENERATION_OUTPUT = (
    REPORT_ROOT / "generation_samples.json"
)

SUMMARY_JSON = (
    REPORT_ROOT
    / "official_benchmark_summary.json"
)

SUMMARY_CSV = (
    REPORT_ROOT
    / "official_benchmark_summary.csv"
)


PROMPTS = (
    "The future of artificial intelligence",
    "Science is important because",
    "If a box is placed on a table",
    "A young researcher discovered that",
)


def repetition_statistics(
    token_ids: list[int],
) -> dict:
    if not token_ids:
        return {
            "tokens": 0,
            "unique_token_ratio": 0.0,
            "repeated_4gram_ratio": 0.0,
        }

    unique_ratio = (
        len(set(token_ids))
        / len(token_ids)
    )

    fourgrams = [
        tuple(
            token_ids[index:index + 4]
        )
        for index in range(
            max(0, len(token_ids) - 3)
        )
    ]

    if fourgrams:
        repeated_ratio = (
            len(fourgrams)
            - len(set(fourgrams))
        ) / len(fourgrams)
    else:
        repeated_ratio = 0.0

    return {
        "tokens": len(token_ids),
        "unique_token_ratio": unique_ratio,
        "repeated_4gram_ratio": (
            repeated_ratio
        ),
    }


generation_samples = []

for prompt_index, prompt in enumerate(
    PROMPTS
):
    torch.manual_seed(
        SEED + prompt_index
    )
    torch.cuda.manual_seed_all(
        SEED + prompt_index
    )

    completion = anka_lm._generate_one(
        prompt,
        {
            "max_gen_toks": 128,
            "do_sample": True,
            "temperature": 0.8,
            "top_p": 0.92,
        },
    )

    generated_ids = anka_lm.tok_encode(
        completion
    )

    sample = {
        "prompt": prompt,
        "completion": completion,
        **repetition_statistics(
            generated_ids
        ),
    }

    generation_samples.append(sample)

    print("\n" + "-" * 80)
    print("PROMPT:", prompt)
    print("OUTPUT:", completion)

atomic_json_save(
    GENERATION_OUTPUT,
    {
        "checkpoint": str(
            FINAL_CHECKPOINT
        ),
        "seed": SEED,
        "temperature": 0.8,
        "top_p": 0.92,
        "samples": generation_samples,
    },
)


def metric_value(
    metrics: dict,
    metric_name: str,
):
    possible_keys = (
        metric_name,
        f"{metric_name},none",
    )

    for key in possible_keys:
        value = metrics.get(key)

        if isinstance(
            value,
            (int, float, np.number),
        ):
            return float(value)

    for key, value in metrics.items():
        if (
            key.split(",")[0]
            == metric_name
            and isinstance(
                value,
                (int, float, np.number),
            )
        ):
            return float(value)

    return None


summary_rows = []

for task_name in OFFICIAL_TASKS:
    task_path = (
        TASK_ROOT
        / f"{task_name}_zero_shot.json"
    )

    assert task_path.exists(), (
        f"missing result: {task_path}"
    )

    with task_path.open(
        "r",
        encoding="utf-8",
    ) as file:
        task_payload = json.load(file)

    metrics = task_payload[
        "harness"
    ]["results"][task_name]

    summary_rows.append(
        {
            "task": task_name,
            "acc": metric_value(
                metrics,
                "acc",
            ),
            "acc_norm": metric_value(
                metrics,
                "acc_norm",
            ),
            "acc_stderr": metric_value(
                metrics,
                "acc_stderr",
            ),
            "acc_norm_stderr": metric_value(
                metrics,
                "acc_norm_stderr",
            ),
            "elapsed_seconds": (
                task_payload[
                    "elapsed_seconds"
                ]
            ),
        }
    )


architecture_statistics = []

for parameter_name, parameter in (
    benchmark_model.named_parameters()
):
    lower_name = parameter_name.lower()

    if any(
        keyword in lower_name
        for keyword in (
            "reflect",
            "morph",
            "router",
            "gate",
        )
    ):
        values = (
            parameter.detach()
            .float()
            .cpu()
        )

        architecture_statistics.append(
            {
                "name": parameter_name,
                "shape": list(values.shape),
                "mean": float(values.mean()),
                "std": float(
                    values.std(
                        unbiased=False
                    )
                ),
                "minimum": float(values.min()),
                "maximum": float(values.max()),
            }
        )


final_summary = {
    "status": "PASS",
    "model": "ANKA-50M RMW3",
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "checkpoint": str(
        FINAL_CHECKPOINT
    ),
    "parameters": PARAMETER_COUNT,
    "parameter_limit": 50_000_000,
    "tokenizer_vocab_size": (
        VOCAB_SIZE
    ),
    "context_length": 1024,
    "num_fewshot": NUM_FEWSHOT,
    "training": TRAINING_METADATA,
    "benchmarks": summary_rows,
    "wikitext103": WIKITEXT_RESULT,
    "generation_file": str(
        GENERATION_OUTPUT
    ),
    "architecture_statistics": (
        architecture_statistics
    ),
    "environment": {
        "gpu": torch.cuda.get_device_name(0),
        "torch": torch.__version__,
        "lm_eval": version("lm_eval"),
        "seed": SEED,
        "dtype": "bfloat16",
    },
}

atomic_json_save(
    SUMMARY_JSON,
    final_summary,
)

with SUMMARY_CSV.open(
    "w",
    encoding="utf-8",
    newline="",
) as file:
    writer = csv.DictWriter(
        file,
        fieldnames=(
            "task",
            "acc",
            "acc_norm",
            "acc_stderr",
            "acc_norm_stderr",
            "elapsed_seconds",
        ),
    )

    writer.writeheader()
    writer.writerows(summary_rows)


print("\n" + "=" * 96)
print("ANKA-50M RMW3 — OFFICIAL BENCHMARK SUMMARY")
print("=" * 96)

for row in summary_rows:
    print(
        f"{row['task']:12s} | "
        f"acc={row['acc']} | "
        f"acc_norm={row['acc_norm']}"
    )

print(
    f"WikiText-103 | "
    f"loss={WIKITEXT_RESULT['mean_loss']:.6f} | "
    f"ppl={WIKITEXT_RESULT['perplexity']:.3f}"
)

print("\nJSON:", SUMMARY_JSON)
print("CSV :", SUMMARY_CSV)
print("\nALL BENCHMARKS COMPLETED: PASS")

## 12. Stage-2 preflight: checkpoint, tokenizer, and corpus structure

Prepares or validates persistent data artifacts. Review the Drive paths before running; large corpus operations can take time and may replace files at the configured destination.

> **Post-release experiment:** this Stage-2 section is not part of the released 5B base checkpoint.


In [ ]:
# STAGE-2 — cell 1
# Checkpoint, tokenizer and corpus structure check does.
# Does not modify or delete any files.

from pathlib import Path
import json

from google.colab import drive


# ============================================================
# 1. DRIVE
# ============================================================

drive.mount(
    "/content/drive",
    force_remount=False,
)


# ============================================================
# 2. PATHS
# ============================================================

RUN_ROOT = Path(
    "/content/drive/MyDrive/ANKA/run_02_english_5b"
)

TOKENIZER_PATH = (
    RUN_ROOT
    / "tokenizer"
    / "tokenizer.json"
)

BASE_CHECKPOINT = (
    RUN_ROOT
    / "rmw3_full_5b_v3"
    / "final.pt"
)

NATURAL_ROOT = (
    RUN_ROOT
    / "natural_4p8b_v3"
)

WORLD_ROOT = (
    RUN_ROOT
    / "worldpairs_200m_v3"
)

STAGE2_ROOT = (
    RUN_ROOT
    / "rmw3_stage2_500m_v1"
)


# ============================================================
# 3. YARDIMCI FONKSIYONLAR
# ============================================================

def format_size(byte_count):
    value = float(byte_count)

    for unit in (
        "B",
        "KiB",
        "MiB",
        "GiB",
        "TiB",
    ):
        if value < 1024.0:
            return f"{value:,.2f} {unit}"

        value /= 1024.0

    return f"{value:,.2f} PiB"


def collect_files(root):
    return [
        path
        for path in root.rglob("*")
        if path.is_file()
    ]


def summarize_files(files):
    total_bytes = 0
    extensions = {}

    for path in files:
        try:
            file_size = path.stat().st_size
        except OSError:
            continue

        total_bytes += file_size

        suffix = (
            path.suffix.lower()
            if path.suffix
            else "<uzantisiz>"
        )

        if suffix not in extensions:
            extensions[suffix] = {
                "count": 0,
                "bytes": 0,
            }

        extensions[suffix]["count"] += 1
        extensions[suffix]["bytes"] += file_size

    return total_bytes, extensions


def collect_token_fields(
    value,
    prefix="",
    results=None,
):
    if results is None:
        results = []

    if isinstance(value, dict):
        for key, item in value.items():
            current_name = (
                f"{prefix}.{key}"
                if prefix
                else str(key)
            )

            normalized_key = (
                str(key)
                .lower()
                .replace("-", "_")
                .replace(" ", "_")
            )

            if (
                "token" in normalized_key
                and isinstance(item, (int, float))
                and not isinstance(item, bool)
            ):
                results.append(
                    (
                        current_name,
                        item,
                    )
                )

            collect_token_fields(
                item,
                current_name,
                results,
            )

    elif isinstance(value, list):
        for index, item in enumerate(
            value[:2000]
        ):
            collect_token_fields(
                item,
                f"{prefix}[{index}]",
                results,
            )

    return results


def find_manifest_files(root):
    json_files = sorted(
        root.rglob("*.json")
    )

    preferred_files = [
        path
        for path in json_files
        if any(
            word in path.name.lower()
            for word in (
                "manifest",
                "metadata",
                "summary",
                "stats",
            )
        )
    ]

    if preferred_files:
        return preferred_files

    return json_files


def inspect_manifest(path):
    print()
    print("Manifest:", path)

    try:
        with path.open(
            "r",
            encoding="utf-8",
        ) as file:
            data = json.load(file)
    except Exception as error:
        print(
            "Manifest okunamadi:",
            repr(error),
        )
        return

    if isinstance(data, dict):
        print(
            "top-level seviye alanlar:",
            list(data.keys())[:40],
        )
    elif isinstance(data, list):
        print(
            "Manifest a liste.",
            "item count:",
            len(data),
        )
    else:
        print(
            "Manifest data tipi:",
            type(data).__name__,
        )

    token_fields = collect_token_fields(
        data
    )

    if not token_fields:
        print(
            "numerical token field not found."
        )
        return

    print("Token alanlari:")

    for field_name, value in token_fields[:80]:
        if isinstance(value, int):
            shown_value = f"{value:,}"
        else:
            shown_value = f"{value:,.6f}"

        print(
            f"  {field_name}: {shown_value}"
        )


# ============================================================
# 4. CHECK REQUIRED FILES
# ============================================================

required_paths = {
    "Tokenizer": TOKENIZER_PATH,
    "Base checkpoint": BASE_CHECKPOINT,
    "Natural corpus": NATURAL_ROOT,
    "WorldPairs corpus": WORLD_ROOT,
}

print("=" * 88)
print("STAGE-2 ON DENETIMI")
print("=" * 88)

missing_paths = []

for name, path in required_paths.items():
    if path.exists():
        status = "found"
    else:
        status = "missing"
        missing_paths.append(
            f"{name}: {path}"
        )

    print(f"{name:<20}: {status}")
    print(f"path                  : {path}")
    print()

if missing_paths:
    raise FileNotFoundError(
        "missing file or directory var:\n"
        + "\n".join(missing_paths)
    )


# ============================================================
# 5. CHECKPOINT and TOKENIZER
# ============================================================

print("=" * 88)
print("MODEL files")
print("=" * 88)

print(
    "Checkpoint size:",
    format_size(
        BASE_CHECKPOINT.stat().st_size
    ),
)

print(
    "Tokenizer size :",
    format_size(
        TOKENIZER_PATH.stat().st_size
    ),
)


# ============================================================
# 6. CHECK CORPORA
# ============================================================

corpus_files = {}

for corpus_name, corpus_root in (
    (
        "natural CORPUS",
        NATURAL_ROOT,
    ),
    (
        "WORLDPAIRS",
        WORLD_ROOT,
    ),
):
    print()
    print("=" * 88)
    print(corpus_name)
    print("=" * 88)
    print("path:", corpus_root)

    files = collect_files(
        corpus_root
    )

    corpus_files[corpus_name] = files

    total_bytes, extensions = (
        summarize_files(files)
    )

    print(
        "file count:",
        f"{len(files):,}",
    )

    print(
        "Total boyut:",
        format_size(total_bytes),
    )

    print("file types:")

    sorted_extensions = sorted(
        extensions.items(),
        key=lambda item: item[1]["bytes"],
        reverse=True,
    )

    for suffix, information in sorted_extensions:
        print(
            f"  {suffix:<12} "
            f"{information['count']:>6,} file | "
            f"{format_size(information['bytes'])}"
        )

    manifests = find_manifest_files(
        corpus_root
    )

    print(
        "Manifest count:",
        len(manifests),
    )

    for manifest_path in manifests[:30]:
        inspect_manifest(
            manifest_path
        )


# ============================================================
# 7. DISCOVER SOURCE FILES
# ============================================================

source_patterns = {
    "fineweb_edu": (
        "fineweb",
        "fineweb_edu",
    ),
    "project_gutenberg": (
        "gutenberg",
        "project_gutenberg",
    ),
    "wikimedia": (
        "wikimedia",
        "wikipedia",
    ),
    "stackexchange": (
        "stackexchange",
        "stack_exchange",
    ),
    "openstax": (
        "openstax",
        "cosmopedia_openstax",
    ),
    "khanacademy": (
        "khanacademy",
        "khan_academy",
        "khan",
    ),
    "libretexts": (
        "libretexts",
        "libretext",
    ),
    "finemath": (
        "finemath",
    ),
    "pes2o": (
        "pes2o",
    ),
    "pubmed": (
        "pubmed",
    ),
}

natural_files = corpus_files[
    "natural CORPUS"
]

print()
print("=" * 88)
print("STAGE-2 source search")
print("=" * 88)

for source_name, patterns in source_patterns.items():
    matched_files = []

    for path in natural_files:
        relative_name = str(
            path.relative_to(
                NATURAL_ROOT
            )
        ).lower()

        if any(
            pattern in relative_name
            for pattern in patterns
        ):
            matched_files.append(path)

    matched_bytes = sum(
        path.stat().st_size
        for path in matched_files
    )

    print()
    print(
        f"{source_name:<20}: "
        f"{len(matched_files):>5,} file | "
        f"{format_size(matched_bytes)}"
    )

    for path in matched_files[:5]:
        print(
            "  ",
            path.relative_to(
                NATURAL_ROOT
            ),
        )

    if len(matched_files) > 5:
        print(
            "  ...",
            f"{len(matched_files) - 5:,}",
            "file more",
        )


# ============================================================
# 8. TOP-LEVEL DIRECTORY STRUCTURE
# ============================================================

print()
print("=" * 88)
print("natural CORPUS TOP-LEVEL DIRECTORY STRUCTURE")
print("=" * 88)

for path in sorted(
    NATURAL_ROOT.iterdir(),
    key=lambda item: item.name.lower(),
):
    if path.is_dir():
        sub_files = collect_files(path)

        sub_size = sum(
            file_path.stat().st_size
            for file_path in sub_files
        )

        print(
            f"[directory] {path.name:<34} "
            f"{len(sub_files):>5,} file | "
            f"{format_size(sub_size)}"
        )
    else:
        print(
            f"[file ] {path.name:<34} "
            f"{format_size(path.stat().st_size)}"
        )


# ============================================================
# 9. RESULT
# ============================================================

print()
print("=" * 88)
print("PREFLIGHT AUDIT RESULT")
print("=" * 88)

print(
    "Stage-2 output path:",
    STAGE2_ROOT,
)

print(
    "Stage-2 directory    :",
    (
        "existing"
        if STAGE2_ROOT.exists()
        else "HENUZ OLUSTURULMADI"
    ),
)

print()
print(
    "original 5B checkpoint "
    "was not modified."
)

print(
    "STAGE-2 ON DENETIMI completed: PASS"
)

## 13. Stage-2 source and shard inventory

This is a compute-intensive stage. Confirm the selected runtime, input paths, token budget, and resume directory before execution. Checkpoints are written incrementally.

> **Post-release experiment:** this Stage-2 section is not part of the released 5B base checkpoint.


In [ ]:
# STAGE-2 — cell 2
# Shows the source report and shard structure in the manifest.
# Does not modify any files.

from pathlib import Path
import json


# ============================================================
# 1. PATHS
# ============================================================

NATURAL_ROOT = Path(
    "/content/drive/MyDrive/ANKA/"
    "run_02_english_5b/"
    "natural_4p8b_v3"
)

MANIFEST_PATH = (
    NATURAL_ROOT
    / "manifest.json"
)

BUILD_STATE_PATH = (
    NATURAL_ROOT
    / "build_state.json"
)


# ============================================================
# 2. files OKU
# ============================================================

if not MANIFEST_PATH.exists():
    raise FileNotFoundError(
        f"Manifest not found: {MANIFEST_PATH}"
    )

with MANIFEST_PATH.open(
    "r",
    encoding="utf-8",
) as file:
    manifest = json.load(file)

if BUILD_STATE_PATH.exists():
    with BUILD_STATE_PATH.open(
        "r",
        encoding="utf-8",
    ) as file:
        build_state = json.load(file)
else:
    build_state = None


# ============================================================
# 3. BASIC MANIFEST INFORMATION
# ============================================================

print("=" * 88)
print("natural CORPUS MANIFEST OZETI")
print("=" * 88)

print(
    "Surum             :",
    manifest.get("version"),
)

print(
    "data tipi         :",
    manifest.get("dtype"),
)

print(
    "Vocabulary        :",
    manifest.get("vocab_size"),
)

print(
    "Train token       :",
    f"{int(manifest.get('train_tokens', 0)):,}",
)

print(
    "Validation token  :",
    f"{int(manifest.get('validation_tokens', 0)):,}",
)

train_shards = manifest.get(
    "train_shards",
    [],
)

validation_shards = manifest.get(
    "validation_shards",
    [],
)

print(
    "Train shard       :",
    len(train_shards),
)

print(
    "Validation shard  :",
    len(validation_shards),
)


# ============================================================
# 4. SOURCE REPORT
# ============================================================

source_report = manifest.get(
    "source_report"
)

print()
print("=" * 88)
print("SOURCE REPORT")
print("=" * 88)

if source_report is None:
    print(
        "Manifest in source_report not found."
    )
else:
    print(
        json.dumps(
            source_report,
            indent=2,
            ensure_ascii=False,
        )
    )


# ============================================================
# 5. INSPECT TRAIN SHARD RECORDS
# ============================================================

print()
print("=" * 88)
print("TRAIN SHARD record examples")
print("=" * 88)

if not train_shards:
    print(
        "Train shard record not found."
    )
else:
    sample_indices = sorted(
        set(
            [
                0,
                1,
                2,
                len(train_shards) // 4,
                len(train_shards) // 2,
                3 * len(train_shards) // 4,
                len(train_shards) - 3,
                len(train_shards) - 2,
                len(train_shards) - 1,
            ]
        )
    )

    for index in sample_indices:
        if not (
            0 <= index < len(train_shards)
        ):
            continue

        print()
        print(
            f"Shard index: {index}"
        )

        print(
            json.dumps(
                train_shards[index],
                indent=2,
                ensure_ascii=False,
            )
        )


# ============================================================
# 6. DISCOVER SOURCE FIELDS IN SHARD RECORDS
# ============================================================

possible_source_keys = (
    "source",
    "source_name",
    "source_key",
    "dataset",
    "dataset_name",
    "category",
)

detected_source_key = None

for shard in train_shards:
    if not isinstance(shard, dict):
        continue

    for key in possible_source_keys:
        if key in shard:
            detected_source_key = key
            break

    if detected_source_key is not None:
        break

print()
print("=" * 88)
print("SHARD source boundaries")
print("=" * 88)

if detected_source_key is None:
    print(
        "The shard records do not directly contain source "
        "field not found."
    )
    print(
        "source boundaries source_report "
        "will be calculated from metadata."
    )
else:
    print(
        "found source field:",
        detected_source_key,
    )

    previous_source = None
    range_start = 0
    accumulated_tokens = 0

    for index, shard in enumerate(
        train_shards
    ):
        current_source = shard.get(
            detected_source_key,
            "<bilinmiyor>",
        )

        shard_tokens = int(
            shard.get(
                "tokens",
                0,
            )
        )

        if previous_source is None:
            previous_source = current_source
            range_start = index

        if current_source != previous_source:
            print(
                f"{previous_source}: "
                f"shard {range_start}–{index - 1} | "
                f"{accumulated_tokens:,} token"
            )

            previous_source = current_source
            range_start = index
            accumulated_tokens = 0

        accumulated_tokens += shard_tokens

    if previous_source is not None:
        print(
            f"{previous_source}: "
            f"shard {range_start}–"
            f"{len(train_shards) - 1} | "
            f"{accumulated_tokens:,} token"
        )


# ============================================================
# 7. BUILD STATE
# ============================================================

print()
print("=" * 88)
print("BUILD STATE")
print("=" * 88)

if build_state is None:
    print(
        "build_state.json not found."
    )
else:
    print(
        json.dumps(
            build_state,
            indent=2,
            ensure_ascii=False,
        )
    )


# ============================================================
# 8. RESULT
# ============================================================

print()
print("=" * 88)
print("STAGE-2 source DENETIMI completed: PASS")
print("=" * 88)

## 14. Stage-2 mixture-capacity calculation

This is a compute-intensive stage. Confirm the selected runtime, input paths, token budget, and resume directory before execution. Checkpoints are written incrementally.

> **Post-release experiment:** this Stage-2 section is not part of the released 5B base checkpoint.


In [ ]:
# STAGE-2 — cell 3
# calculates exact source ranges in the corpus.
# 500M-token Stage-2 builds the mixture plan.
# The corpus and main checkpoint are not modified.

from pathlib import Path
from bisect import bisect_left, bisect_right
import json
import math


# ============================================================
# 1. PATHS
# ============================================================

RUN_ROOT = Path(
    "/content/drive/MyDrive/ANKA/"
    "run_02_english_5b"
)

NATURAL_ROOT = (
    RUN_ROOT
    / "natural_4p8b_v3"
)

WORLD_ROOT = (
    RUN_ROOT
    / "worldpairs_200m_v3"
)

STAGE2_ROOT = (
    RUN_ROOT
    / "rmw3_stage2_500m_v1"
)

NATURAL_MANIFEST_PATH = (
    NATURAL_ROOT
    / "manifest.json"
)

WORLD_MANIFEST_PATH = (
    WORLD_ROOT
    / "manifest.json"
)

PLAN_PATH = (
    STAGE2_ROOT
    / "stage2_data_plan.json"
)


# ============================================================
# 2. MANIFESTLERI OKU
# ============================================================

if not NATURAL_MANIFEST_PATH.exists():
    raise FileNotFoundError(
        f"missing manifest: {NATURAL_MANIFEST_PATH}"
    )

if not WORLD_MANIFEST_PATH.exists():
    raise FileNotFoundError(
        f"missing manifest: {WORLD_MANIFEST_PATH}"
    )

with NATURAL_MANIFEST_PATH.open(
    "r",
    encoding="utf-8",
) as file:
    natural_manifest = json.load(file)

with WORLD_MANIFEST_PATH.open(
    "r",
    encoding="utf-8",
) as file:
    world_manifest = json.load(file)


# ============================================================
# 3. CALCULATE SHARD RANGES
# ============================================================

train_shards = natural_manifest.get(
    "train_shards",
    [],
)

if not train_shards:
    raise RuntimeError(
        "Natural corpus train shard listesi empty."
    )

shard_starts = []
shard_ends = []

shard_cursor = 0

for shard in train_shards:
    shard_tokens = int(
        shard["tokens"]
    )

    shard_starts.append(
        shard_cursor
    )

    shard_cursor += shard_tokens

    shard_ends.append(
        shard_cursor
    )

manifest_train_tokens = int(
    natural_manifest["train_tokens"]
)

if shard_cursor != manifest_train_tokens:
    raise RuntimeError(
        "Shard token total manifest with does-not-match. "
        f"Shard={shard_cursor:,}, "
        f"manifest={manifest_train_tokens:,}"
    )


# ============================================================
# 4. CALCULATE SOURCE RANGES
# ============================================================

source_report = natural_manifest.get(
    "source_report",
    {},
)

if not source_report:
    raise RuntimeError(
        "Manifest in source_report unavailable."
    )

source_ranges = {}

source_cursor = 0

for source_name, report in source_report.items():
    source_tokens = int(
        report.get(
            "final_train_tokens",
            report.get(
                "train_tokens",
                0,
            ),
        )
    )

    source_start = source_cursor
    source_end = (
        source_start
        + source_tokens
    )

    start_shard = bisect_right(
        shard_ends,
        source_start,
    )

    end_shard = bisect_left(
        shard_ends,
        source_end,
    )

    source_ranges[source_name] = {
        "start_token": source_start,
        "end_token": source_end,
        "tokens": source_tokens,
        "start_shard": start_shard,
        "end_shard": end_shard,
    }

    source_cursor = source_end

if source_cursor != manifest_train_tokens:
    raise RuntimeError(
        "source token total manifest with does-not-match. "
        f"source={source_cursor:,}, "
        f"manifest={manifest_train_tokens:,}"
    )


# ============================================================
# 5. STAGE-2 DATA MIX
# ============================================================

# Goal:
# - general language bilgisini korumak
# - HellaSwag for event ordering and narrative
# - WinoGrande for long-range links and references
# - PIQA for procedural/physical text
# - ARC-Easy preserve performance
# - WorldPairs continue relational learning

STAGE2_MIX = [
    {
        "name": "general_replay",
        "source": "fineweb_edu_fill",
        "tokens": 325_000_000,
        "purpose": "general capabilities and validation loss'u preservation",
    },
    {
        "name": "narrative_reasoning",
        "source": "project_gutenberg",
        "tokens": 60_000_000,
        "purpose": "HellaSwag event ordering and narrative completion",
    },
    {
        "name": "reference_reasoning",
        "source": "wikimedia",
        "tokens": 40_000_000,
        "purpose": "WinoGrande referans and context takibi",
    },
    {
        "name": "procedural_physical",
        "source": "stackexchange_general",
        "tokens": 30_000_000,
        "purpose": "PIQA prosedurel and physical reasoning",
    },
    {
        "name": "science_openstax",
        "source": "cosmopedia_openstax",
        "tokens": 10_000_000,
        "purpose": "ARC-Easy science knowledge",
    },
    {
        "name": "science_khan",
        "source": "cosmopedia_khanacademy",
        "tokens": 5_000_000,
        "purpose": "Temel training and explanatory metin",
    },
    {
        "name": "science_libretexts",
        "source": "libretexts_general",
        "tokens": 10_000_000,
        "purpose": "Ders kitabi knowledge and science",
    },
    {
        "name": "worldpairs",
        "source": "worldpairs",
        "tokens": 20_000_000,
        "purpose": "relational and counterfactual state learning",
    },
]

TARGET_STAGE2_TOKENS = sum(
    item["tokens"]
    for item in STAGE2_MIX
)

if TARGET_STAGE2_TOKENS != 500_000_000:
    raise RuntimeError(
        "Stage-2 plan 500M token does-not-total. "
        f"Total={TARGET_STAGE2_TOKENS:,}"
    )


# ============================================================
# 6. VERIFY SOURCE CAPACITIES
# ============================================================

world_tokens = int(
    world_manifest["tokens"]
)

resolved_mix = []

for item in STAGE2_MIX:
    source_name = item["source"]
    requested_tokens = int(
        item["tokens"]
    )

    if source_name == "worldpairs":
        available_tokens = world_tokens

        resolved_item = {
            **item,
            "dataset": "worldpairs",
            "available_tokens": available_tokens,
            "start_token": 0,
            "end_token": world_tokens,
            "start_shard": 0,
            "end_shard": (
                len(
                    world_manifest.get(
                        "shards",
                        [],
                    )
                )
                - 1
            ),
        }
    else:
        if source_name not in source_ranges:
            raise KeyError(
                f"source not found: {source_name}"
            )

        source_information = source_ranges[
            source_name
        ]

        available_tokens = int(
            source_information["tokens"]
        )

        resolved_item = {
            **item,
            "dataset": "natural",
            "available_tokens": available_tokens,
            "start_token": int(
                source_information[
                    "start_token"
                ]
            ),
            "end_token": int(
                source_information[
                    "end_token"
                ]
            ),
            "start_shard": int(
                source_information[
                    "start_shard"
                ]
            ),
            "end_shard": int(
                source_information[
                    "end_shard"
                ]
            ),
        }

    if requested_tokens > available_tokens:
        raise RuntimeError(
            f"{source_name} for yeterli data unavailable. "
            f"Istenen={requested_tokens:,}, "
            f"existing={available_tokens:,}"
        )

    resolved_item["probability"] = (
        requested_tokens
        / TARGET_STAGE2_TOKENS
    )

    resolved_mix.append(
        resolved_item
    )


# ============================================================
# 7. TRAINING STEP CALCULATION
# ============================================================

BATCH_SIZE = 64
SEQUENCE_LENGTH = 1024

TOKENS_PER_STEP = (
    BATCH_SIZE
    * SEQUENCE_LENGTH
)

TOTAL_STEPS = math.ceil(
    TARGET_STAGE2_TOKENS
    / TOKENS_PER_STEP
)

ACTUAL_TRAINING_TOKENS = (
    TOTAL_STEPS
    * TOKENS_PER_STEP
)


# ============================================================
# 8. SAVE PLAN TO DRIVE
# ============================================================

STAGE2_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

plan = {
    "version": 1,
    "base_checkpoint": str(
        RUN_ROOT
        / "rmw3_full_5b_v3"
        / "final.pt"
    ),
    "tokenizer": str(
        RUN_ROOT
        / "tokenizer"
        / "tokenizer.json"
    ),
    "natural_manifest": str(
        NATURAL_MANIFEST_PATH
    ),
    "world_manifest": str(
        WORLD_MANIFEST_PATH
    ),
    "target_tokens": (
        TARGET_STAGE2_TOKENS
    ),
    "actual_training_tokens": (
        ACTUAL_TRAINING_TOKENS
    ),
    "batch_size": BATCH_SIZE,
    "sequence_length": SEQUENCE_LENGTH,
    "tokens_per_step": TOKENS_PER_STEP,
    "total_steps": TOTAL_STEPS,
    "mix": resolved_mix,
    "all_natural_source_ranges": (
        source_ranges
    ),
}

with PLAN_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        plan,
        file,
        indent=2,
        ensure_ascii=False,
    )


# ============================================================
# 9. PRINT RESULT
# ============================================================

print("=" * 108)
print("ANKA RMW3 — STAGE-2 data plan")
print("=" * 108)

print(
    f"{'section':<24}"
    f"{'source':<28}"
    f"{'Planlanan':>15}"
    f"{'ratio':>10}"
    f"{'existing':>15}"
)

print("-" * 108)

for item in resolved_mix:
    print(
        f"{item['name']:<24}"
        f"{item['source']:<28}"
        f"{item['tokens']:>15,}"
        f"{item['probability'] * 100:>9.1f}%"
        f"{item['available_tokens']:>15,}"
    )

print("-" * 108)

print(
    "target token       :",
    f"{TARGET_STAGE2_TOKENS:,}",
)

print(
    "actual token      :",
    f"{ACTUAL_TRAINING_TOKENS:,}",
)

print(
    "Batch             :",
    BATCH_SIZE,
)

print(
    "Sequence          :",
    SEQUENCE_LENGTH,
)

print(
    "Token/step        :",
    f"{TOKENS_PER_STEP:,}",
)

print(
    "Total step       :",
    f"{TOTAL_STEPS:,}",
)

print(
    "Plan filesi      :",
    PLAN_PATH,
)

print()
print(
    "The base 5B checkpoint was not modified."
)

print(
    "STAGE-2 data plan: PASS"
)

## 15. Stage-2 new-source access and tokenizer preflight

Prepares or validates persistent data artifacts. Review the Drive paths before running; large corpus operations can take time and may replace files at the configured destination.

> **Post-release experiment:** this Stage-2 section is not part of the released 5B base checkpoint.


In [ ]:
# STAGE-2 — cell 4
# Quick access and tokenizer test for new data sources.
# The main corpus and checkpoint are not modified.

from pathlib import Path
import importlib.util
import subprocess
import sys
import json
import time


# ============================================================
# 1. required PAKETLER
# ============================================================

required_packages = {
    "datasets": "datasets",
    "tokenizers": "tokenizers",
}

missing_packages = [
    pip_name
    for module_name, pip_name
    in required_packages.items()
    if importlib.util.find_spec(module_name) is None
]

if missing_packages:
    print(
        "missing paketler kuruluyor:",
        missing_packages,
    )

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            *missing_packages,
        ],
        check=True,
    )


# Paket kurulumu after-completion after import do.

from datasets import load_dataset
from tokenizers import Tokenizer
from google.colab import drive


# ============================================================
# 2. DRIVE AND PATHS
# ============================================================

drive.mount(
    "/content/drive",
    force_remount=False,
)

RUN_ROOT = Path(
    "/content/drive/MyDrive/ANKA/"
    "run_02_english_5b"
)

TOKENIZER_PATH = (
    RUN_ROOT
    / "tokenizer"
    / "tokenizer.json"
)

STAGE2_ROOT = (
    RUN_ROOT
    / "rmw3_stage2_500m_v1"
)

PREFLIGHT_PATH = (
    STAGE2_ROOT
    / "new_data_preflight.json"
)

if not TOKENIZER_PATH.exists():
    raise FileNotFoundError(
        f"Tokenizer not found: {TOKENIZER_PATH}"
    )

STAGE2_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 3. TOKENIZER
# ============================================================

tokenizer = Tokenizer.from_file(
    str(TOKENIZER_PATH)
)

vocabulary_size = tokenizer.get_vocab_size()

if vocabulary_size != 16_384:
    raise RuntimeError(
        "Tokenizer vocabulary size expectedden different. "
        f"found={vocabulary_size:,}, expected=16,384"
    )

UNK_ID = 3


# ============================================================
# 4. NEW DATA SOURCES
# ============================================================

NEW_DATA_SOURCES = [
    {
        "key": "fineweb_edu_dedup_new",
        "repo": "HuggingFaceTB/smollm-corpus",
        "config": "fineweb-edu-dedup",
        "split": "train",
        "text_fields": ["text"],
        "target_tokens": 180_000_000,
        "future_skip_rows": 0,
        "shuffle_seed": 20_260_922,
        "purpose": "general language, well-formed sentence and training metni",
    },
    {
        "key": "pg19_new",
        "repo": "emozilla/pg19",
        "config": None,
        "split": "train",
        "text_fields": ["text"],
        "target_tokens": 100_000_000,
        "future_skip_rows": 2_000,
        "shuffle_seed": 20_260_923,
        "purpose": "narrative, language knowledge and long context",
    },
    {
        "key": "open_web_math_new",
        "repo": "open-web-math/open-web-math",
        "config": None,
        "split": "train",
        "text_fields": ["text"],
        "target_tokens": 60_000_000,
        "future_skip_rows": 0,
        "shuffle_seed": 20_260_924,
        "purpose": "mathematics and bilimsel reasoning",
    },
    {
        "key": "stackexchange_new",
        "repo": "common-pile/stackexchange",
        "config": None,
        "split": "train",
        "text_fields": ["text"],
        "target_tokens": 40_000_000,
        "future_skip_rows": 531_221,
        "shuffle_seed": None,
        "purpose": "Prosedurel and physical explanations",
    },
]


# ============================================================
# 5. TEXT HELPERS
# ============================================================

def extract_text(row, preferred_fields):
    for field_name in preferred_fields:
        value = row.get(field_name)

        if isinstance(value, str):
            value = value.strip()

            if value:
                return value

    for value in row.values():
        if isinstance(value, str):
            value = value.strip()

            if len(value) >= 200:
                return value

    return ""


def text_quality_statistics(text):
    total_characters = len(text)

    if total_characters == 0:
        return {
            "ascii_letter_ratio": 0.0,
            "line_repetition_ratio": 1.0,
        }

    letters = [
        character
        for character in text
        if character.isalpha()
    ]

    ascii_letters = [
        character
        for character in letters
        if (
            "a" <= character.lower() <= "z"
        )
    ]

    if letters:
        ascii_letter_ratio = (
            len(ascii_letters)
            / len(letters)
        )
    else:
        ascii_letter_ratio = 0.0

    lines = [
        line.strip()
        for line in text.splitlines()
        if line.strip()
    ]

    if lines:
        line_repetition_ratio = (
            1.0
            - len(set(lines)) / len(lines)
        )
    else:
        line_repetition_ratio = 0.0

    return {
        "ascii_letter_ratio": (
            ascii_letter_ratio
        ),
        "line_repetition_ratio": (
            line_repetition_ratio
        ),
    }


def open_streaming_source(source):
    arguments = {
        "path": source["repo"],
        "split": source["split"],
        "streaming": True,
    }

    if source["config"] is not None:
        arguments["name"] = source["config"]

    dataset = load_dataset(
        **arguments
    )

    return dataset


# ============================================================
# 6. PREFLIGHT TEST
# ============================================================

PREFLIGHT_SCAN_ROWS = 32
PREFLIGHT_ACCEPTED_DOCUMENTS = 12
MAX_CHARACTERS_PER_DOCUMENT = 50_000

preflight_results = []
failed_sources = []

for source_index, source in enumerate(
    NEW_DATA_SOURCES,
    start=1,
):
    print()
    print("=" * 88)
    print(
        f"[{source_index:02d}/"
        f"{len(NEW_DATA_SOURCES):02d}] "
        f"{source['key']}"
    )
    print("=" * 88)

    print(
        "Repo   :",
        source["repo"],
    )

    print(
        "Config :",
        source["config"],
    )

    print(
        "Goal   :",
        source["purpose"],
    )

    start_time = time.perf_counter()

    try:
        dataset = open_streaming_source(
            source
        )

        iterator = iter(dataset)

        scanned_rows = 0
        accepted_documents = 0
        total_characters = 0
        total_tokens = 0
        unknown_tokens = 0
        ascii_ratio_sum = 0.0
        repetition_ratio_sum = 0.0
        first_text_fields = None

        while (
            scanned_rows < PREFLIGHT_SCAN_ROWS
            and accepted_documents
            < PREFLIGHT_ACCEPTED_DOCUMENTS
        ):
            try:
                row = next(iterator)
            except StopIteration:
                break

            scanned_rows += 1

            if first_text_fields is None:
                first_text_fields = list(
                    row.keys()
                )

            text = extract_text(
                row,
                source["text_fields"],
            )

            if len(text) < 200:
                continue

            text = text[
                :MAX_CHARACTERS_PER_DOCUMENT
            ]

            encoded = tokenizer.encode(
                text
            )

            token_ids = encoded.ids

            if len(token_ids) < 32:
                continue

            quality = (
                text_quality_statistics(
                    text
                )
            )

            accepted_documents += 1
            total_characters += len(text)
            total_tokens += len(token_ids)

            unknown_tokens += sum(
                token_id == UNK_ID
                for token_id in token_ids
            )

            ascii_ratio_sum += quality[
                "ascii_letter_ratio"
            ]

            repetition_ratio_sum += quality[
                "line_repetition_ratio"
            ]

        elapsed_seconds = (
            time.perf_counter()
            - start_time
        )

        if accepted_documents == 0:
            raise RuntimeError(
                "No usable text was found."
            )

        characters_per_token = (
            total_characters
            / total_tokens
        )

        unknown_ratio = (
            unknown_tokens
            / total_tokens
        )

        mean_ascii_ratio = (
            ascii_ratio_sum
            / accepted_documents
        )

        mean_repetition_ratio = (
            repetition_ratio_sum
            / accepted_documents
        )

        result = {
            "key": source["key"],
            "repo": source["repo"],
            "config": source["config"],
            "purpose": source["purpose"],
            "target_tokens": (
                source["target_tokens"]
            ),
            "future_skip_rows": (
                source["future_skip_rows"]
            ),
            "shuffle_seed": (
                source["shuffle_seed"]
            ),
            "scanned_rows": scanned_rows,
            "accepted_documents": (
                accepted_documents
            ),
            "sample_fields": (
                first_text_fields
            ),
            "sample_characters": (
                total_characters
            ),
            "sample_tokens": total_tokens,
            "characters_per_token": (
                characters_per_token
            ),
            "unknown_tokens": (
                unknown_tokens
            ),
            "unknown_ratio": (
                unknown_ratio
            ),
            "mean_ascii_letter_ratio": (
                mean_ascii_ratio
            ),
            "mean_line_repetition_ratio": (
                mean_repetition_ratio
            ),
            "elapsed_seconds": (
                elapsed_seconds
            ),
            "status": "PASS",
        }

        preflight_results.append(
            result
        )

        print(
            "Alanlar        :",
            first_text_fields,
        )

        print(
            "scanned row  :",
            f"{scanned_rows:,}",
        )

        print(
            "Accepted       :",
            f"{accepted_documents:,}",
        )

        print(
            "Example token    :",
            f"{total_tokens:,}",
        )

        print(
            "Karakter/token :",
            f"{characters_per_token:.3f}",
        )

        print(
            "UNK token      :",
            f"{unknown_tokens:,}",
        )

        print(
            "ASCII harf     :",
            f"{mean_ascii_ratio * 100:.2f}%",
        )

        print(
            "row duplicates  :",
            f"{mean_repetition_ratio * 100:.2f}%",
        )

        print(
            "Duration           :",
            f"{elapsed_seconds:.1f} saniye",
        )

        print(
            "RESULT          : PASS"
        )

        del dataset
        del iterator

    except Exception as error:
        failed_sources.append(
            source["key"]
        )

        result = {
            "key": source["key"],
            "repo": source["repo"],
            "config": source["config"],
            "status": "FAIL",
            "error": repr(error),
        }

        preflight_results.append(
            result
        )

        print(
            "RESULT          : FAIL"
        )

        print(
            "error           :",
            repr(error),
        )


# ============================================================
# 7. SAVE RAP REPORT
# ============================================================

preflight_report = {
    "version": 1,
    "tokenizer": str(
        TOKENIZER_PATH
    ),
    "vocab_size": vocabulary_size,
    "sources": preflight_results,
}

with PREFLIGHT_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        preflight_report,
        file,
        indent=2,
        ensure_ascii=False,
    )


# ============================================================
# 8. RESULT
# ============================================================

print()
print("=" * 88)
print("new data PREFLIGHT OZETI")
print("=" * 88)

for result in preflight_results:
    print(
        f"{result['key']:<28}: "
        f"{result['status']}"
    )

print()
print(
    "report:",
    PREFLIGHT_PATH,
)

if failed_sources:
    raise RuntimeError(
        "Preflight failed sources: "
        + ", ".join(failed_sources)
    )

print()
print(
    "all NEW DATA SOURCES: PASS"
)

## 16. Stage-2A architecture-focused mixture plan

This is a compute-intensive stage. Confirm the selected runtime, input paths, token budget, and resume directory before execution. Checkpoints are written incrementally.

> **Post-release experiment:** this Stage-2 section is not part of the released 5B base checkpoint.


In [ ]:
# STAGE-2A — cell 5
# 500M-token architecture-focused pilot training plan.
# Base checkpoint and the earlier corpus is not modified.

from pathlib import Path
import json
import math


# ============================================================
# 1. PATHS
# ============================================================

RUN_ROOT = Path(
    "/content/drive/MyDrive/ANKA/"
    "run_02_english_5b"
)

BASE_CHECKPOINT = (
    RUN_ROOT
    / "rmw3_full_5b_v3"
    / "final.pt"
)

TOKENIZER_PATH = (
    RUN_ROOT
    / "tokenizer"
    / "tokenizer.json"
)

PREFLIGHT_PATH = (
    RUN_ROOT
    / "rmw3_stage2_500m_v1"
    / "new_data_preflight.json"
)

STAGE2A_ROOT = (
    RUN_ROOT
    / "rmw3_arch_stage2a_500m_v1"
)

CORPUS_ROOT = (
    STAGE2A_ROOT
    / "corpus"
)

CHECKPOINT_ROOT = (
    STAGE2A_ROOT
    / "checkpoints"
)

REPORT_ROOT = (
    STAGE2A_ROOT
    / "reports"
)

PLAN_PATH = (
    STAGE2A_ROOT
    / "architecture_training_plan.json"
)


# ============================================================
# 2. BASIC FILE CHECKS
# ============================================================

required_paths = {
    "Base checkpoint": BASE_CHECKPOINT,
    "Tokenizer": TOKENIZER_PATH,
    "Preflight report": PREFLIGHT_PATH,
}

for name, path in required_paths.items():
    if not path.exists():
        raise FileNotFoundError(
            f"{name} not found: {path}"
        )

with PREFLIGHT_PATH.open(
    "r",
    encoding="utf-8",
) as file:
    preflight_report = json.load(file)

failed_sources = [
    source["key"]
    for source in preflight_report["sources"]
    if source["status"] != "PASS"
]

if failed_sources:
    raise RuntimeError(
        "Preflight failed sources var: "
        + ", ".join(failed_sources)
    )


# ============================================================
# 3. NEW STAGE-2A DIRECTORIES
# ============================================================

for directory in (
    STAGE2A_ROOT,
    CORPUS_ROOT,
    CHECKPOINT_ROOT,
    REPORT_ROOT,
):
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# ============================================================
# 4. DATA MIX
# ============================================================

DATA_MIX = [
    {
        "key": "fineweb_edu_dedup_new",
        "dataset_type": "streaming",
        "repo": "HuggingFaceTB/smollm-corpus",
        "config": "fineweb-edu-dedup",
        "split": "train",
        "domain": "clean_general",
        "target_tokens": 180_000_000,
        "shuffle_seed": 20_260_922,
        "skip_rows": 0,
        "purpose": (
            "clean Ingilizce, general bilgi and "
            "language knowledge"
        ),
    },
    {
        "key": "grammar_clean_new",
        "dataset_type": "derived_streaming",
        "repo": "HuggingFaceTB/smollm-corpus",
        "config": "fineweb-edu-dedup",
        "split": "train",
        "domain": "grammar",
        "target_tokens": 20_000_000,
        "shuffle_seed": 20_260_932,
        "skip_rows": 0,
        "purpose": (
            "too clean, full and well-formed "
            "Ingilizce cumleler"
        ),
    },
    {
        "key": "pg19_new",
        "dataset_type": "streaming",
        "repo": "emozilla/pg19",
        "config": None,
        "split": "train",
        "domain": "narrative",
        "target_tokens": 100_000_000,
        "shuffle_seed": 20_260_923,
        "skip_rows": 2_000,
        "purpose": (
            "Hikaye, olay sirasi, long context "
            "and coreference"
        ),
    },
    {
        "key": "open_web_math_new",
        "dataset_type": "streaming",
        "repo": "open-web-math/open-web-math",
        "config": None,
        "split": "train",
        "domain": "math_science",
        "target_tokens": 60_000_000,
        "shuffle_seed": 20_260_924,
        "skip_rows": 0,
        "purpose": (
            "mathematics, science and too adimli "
            "explanations"
        ),
    },
    {
        "key": "stackexchange_new",
        "dataset_type": "streaming_continuation",
        "repo": "common-pile/stackexchange",
        "config": None,
        "split": "train",
        "domain": "procedural_physical",
        "target_tokens": 40_000_000,
        "shuffle_seed": None,
        "skip_rows": 531_221,
        "purpose": (
            "Prosedurel, teknik and physical "
            "explanations"
        ),
    },
    {
        "key": "natural_replay",
        "dataset_type": "local_replay",
        "manifest": str(
            RUN_ROOT
            / "natural_4p8b_v3"
            / "manifest.json"
        ),
        "domain": "replay",
        "target_tokens": 80_000_000,
        "shuffle_seed": 20_260_925,
        "purpose": (
            "existing general capabilities and "
            "validation loss'u preservation"
        ),
    },
    {
        "key": "worldpairs_v2",
        "dataset_type": "generated_pairs",
        "domain": "worldpairs",
        "target_tokens": 20_000_000,
        "generation_seed": 20_260_926,
        "purpose": (
            "Reflect, Morph and pair-ranking "
            "trainingi"
        ),
    },
]


# ============================================================
# 5. TOKEN HESABI
# ============================================================

TARGET_TOKENS = sum(
    source["target_tokens"]
    for source in DATA_MIX
)

if TARGET_TOKENS != 500_000_000:
    raise RuntimeError(
        "Stage-2A data plan 500M token does-not-total. "
        f"found={TARGET_TOKENS:,}"
    )

BATCH_SIZE = 64
SEQUENCE_LENGTH = 1024

TOKENS_PER_STEP = (
    BATCH_SIZE
    * SEQUENCE_LENGTH
)

TOTAL_STEPS = math.ceil(
    TARGET_TOKENS
    / TOKENS_PER_STEP
)

ACTUAL_TOKENS = (
    TOTAL_STEPS
    * TOKENS_PER_STEP
)

for source in DATA_MIX:
    source["mixture_probability"] = (
        source["target_tokens"]
        / TARGET_TOKENS
    )


# ============================================================
# 6. ARCHITECTURE-FOCUSED TRAINING SETTINGS
# ============================================================

ARCHITECTURE_OBJECTIVES = {
    "language_modelng": {
        "enabled": True,
        "weight": 1.0,
    },
    "reflect_refinement": {
        "enabled": True,
        "weight": 0.02,
        "margin": 0.02,
        "description": (
            "The second recurrent pass's token "
            "loss'unu birinci gecisten low "
            "attempts to preserve it."
        ),
    },
    "morph_load_balance": {
        "enabled": True,
        "initial_weight": 0.01,
        "final_weight": 0.003,
        "description": (
            "experts fully collapse "
            "prevents collapse while preserving specialization "
            "zorla unavailable etmez."
        ),
    },
    "worldpairs_ranking": {
        "enabled": True,
        "weight": 0.05,
        "margin": 0.15,
        "description": (
            "correct dizinin loss'unu incorrect "
            "counterfactual diziden low tutar."
        ),
    },
}


# ============================================================
# 7. OPTIMIZATION SETTINGS
# ============================================================

TRAINING_CONFIG = {
    "precision": "bfloat16",
    "batch_size": BATCH_SIZE,
    "sequence_length": SEQUENCE_LENGTH,
    "tokens_per_step": TOKENS_PER_STEP,
    "total_steps": TOTAL_STEPS,
    "target_tokens": TARGET_TOKENS,
    "actual_tokens": ACTUAL_TOKENS,
    "learning_rate": {
        "schedule": "cosine",
        "peak": 2.0e-4,
        "minimum": 2.0e-5,
        "warmup_steps": 150,
    },
    "optimizer": {
        "name": "AdamW",
        "betas": [0.9, 0.95],
        "weight_decay": 0.1,
        "fused": True,
    },
    "gradient_clip": 1.0,
    "log_interval_steps": 10,
    "validation_interval_steps": 750,
    "checkpoint_interval_steps": 750,
    "keep_last_checkpoints": 3,
    "early_stop": {
        "enabled": True,
        "base_best_validation_loss": (
            2.3343789130449295
        ),
        "maximum_allowed_loss": 2.37,
        "consecutive_failures": 2,
    },
}


# ============================================================
# 8. ARCHITECTURE METRICS TO TRACK
# ============================================================

MONITORED_METRICS = [
    "training_loss",
    "validation_loss",
    "gradient_norm",
    "tokens_per_second",
    "data_wait_ms",
    "reflect_gate_mean",
    "reflect_gate_saturation",
    "pass_1_loss",
    "pass_2_loss",
    "reflect_improvement",
    "router_entropy",
    "expert_loads_by_stage",
    "expert_loads_by_domain",
    "worldpairs_correct_loss",
    "worldpairs_wrong_loss",
    "worldpairs_accuracy",
    "worldpairs_margin",
]


# ============================================================
# 9. SAVE PLAN
# ============================================================

training_plan = {
    "version": 1,
    "experiment": "RMW3_ARCH_STAGE2A_500M",
    "base_checkpoint": str(
        BASE_CHECKPOINT
    ),
    "tokenizer": str(
        TOKENIZER_PATH
    ),
    "stage2a_root": str(
        STAGE2A_ROOT
    ),
    "corpus_root": str(
        CORPUS_ROOT
    ),
    "checkpoint_root": str(
        CHECKPOINT_ROOT
    ),
    "report_root": str(
        REPORT_ROOT
    ),
    "data_mix": DATA_MIX,
    "architecture_objectives": (
        ARCHITECTURE_OBJECTIVES
    ),
    "training_config": TRAINING_CONFIG,
    "monitored_metrics": (
        MONITORED_METRICS
    ),
}

with PLAN_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        training_plan,
        file,
        indent=2,
        ensure_ascii=False,
    )


# ============================================================
# 10. PRINT RESULT
# ============================================================

print("=" * 110)
print("ANKA RMW3 — ARCHITECTURE-FOCUSED STAGE-2A plan")
print("=" * 110)

print(
    f"{'data bolumu':<28}"
    f"{'Domain':<24}"
    f"{'Token':>16}"
    f"{'ratio':>12}"
)

print("-" * 110)

for source in DATA_MIX:
    print(
        f"{source['key']:<28}"
        f"{source['domain']:<24}"
        f"{source['target_tokens']:>16,}"
        f"{source['mixture_probability'] * 100:>11.1f}%"
    )

print("-" * 110)

print(
    "target token       :",
    f"{TARGET_TOKENS:,}",
)

print(
    "actual token      :",
    f"{ACTUAL_TOKENS:,}",
)

print(
    "Total step       :",
    f"{TOTAL_STEPS:,}",
)

print(
    "Batch / sequence  :",
    f"{BATCH_SIZE} / {SEQUENCE_LENGTH}",
)

print(
    "Plan filesi      :",
    PLAN_PATH,
)

print(
    "Corpus directory    :",
    CORPUS_ROOT,
)

print(
    "Checkpoint directory:",
    CHECKPOINT_ROOT,
)

print()
print(
    "The base 5B checkpoint was not modified."
)

print(
    "ARCHITECTURE-FOCUSED STAGE-2A plan: PASS"
)

## 17. Stage-2A corpus construction with provenance metadata

This is a compute-intensive stage. Confirm the selected runtime, input paths, token budget, and resume directory before execution. Checkpoints are written incrementally.

> **Destructive path operation:** this cell removes or replaces files at an explicit path. Verify the destination before execution.

> **Post-release experiment:** this Stage-2 section is not part of the released 5B base checkpoint.


In [ ]:
# STAGE-2A — cell 6
# 400M new tokens while preserving source and domain metadata
# uint16 binary shard converts to format.
#
# If interrupted, rerunning the cell resumes from the last completed
# shard resuming from does.
#
# The main corpus and main checkpoint are not modified.

from pathlib import Path
from array import array
import hashlib
import json
import os
import re
import sys
import time

from datasets import load_dataset
from tokenizers import Tokenizer


# ============================================================
# 1. FIXED PATHS
# ============================================================

RUN_ROOT = Path(
    "/content/drive/MyDrive/ANKA/"
    "run_02_english_5b"
)

STAGE2A_ROOT = (
    RUN_ROOT
    / "rmw3_arch_stage2a_500m_v1"
)

PLAN_PATH = (
    STAGE2A_ROOT
    / "architecture_training_plan.json"
)

CORPUS_ROOT = (
    STAGE2A_ROOT
    / "corpus"
)

NEW_DATA_ROOT = (
    CORPUS_ROOT
    / "new_streaming"
)

TOKENIZER_PATH = (
    RUN_ROOT
    / "tokenizer"
    / "tokenizer.json"
)

GLOBAL_HASH_PATH = (
    NEW_DATA_ROOT
    / "accepted_document_hashes.txt"
)

NEW_MANIFEST_PATH = (
    NEW_DATA_ROOT
    / "new_data_manifest.json"
)


# ============================================================
# 2. CORE CHECKS
# ============================================================

if sys.byteorder != "little":
    raise RuntimeError(
        "this binary format little-endian sistem requires."
    )

if not PLAN_PATH.exists():
    raise FileNotFoundError(
        f"Training plan not found: {PLAN_PATH}"
    )

if not TOKENIZER_PATH.exists():
    raise FileNotFoundError(
        f"Tokenizer not found: {TOKENIZER_PATH}"
    )

with PLAN_PATH.open(
    "r",
    encoding="utf-8",
) as file:
    training_plan = json.load(file)

tokenizer = Tokenizer.from_file(
    str(TOKENIZER_PATH)
)

if tokenizer.get_vocab_size() != 16_384:
    raise RuntimeError(
        "Tokenizer vocabulary size 16,384 not."
    )

EOS_ID = 2
SHARD_TARGET_TOKENS = 25_000_000
MAX_ENCODE_CHARACTERS = 200_000

NEW_DATA_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 3. ATOMIC JSON WRITE
# ============================================================

def atomic_write_json(path, value):
    temporary_path = path.with_suffix(
        path.suffix + ".tmp"
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            value,
            file,
            indent=2,
            ensure_ascii=False,
        )

        file.flush()
        os.fsync(
            file.fileno()
        )

    temporary_path.replace(path)


# ============================================================
# 4. GLOBAL DEDUP HASHES
# ============================================================

def load_document_hashes(path):
    if not path.exists():
        return set()

    with path.open(
        "r",
        encoding="utf-8",
    ) as file:
        return {
            line.strip()
            for line in file
            if line.strip()
        }


def append_document_hashes(path, hashes):
    if not hashes:
        return

    with path.open(
        "a",
        encoding="utf-8",
    ) as file:
        for document_hash in hashes:
            file.write(
                document_hash + "\n"
            )

        file.flush()
        os.fsync(
            file.fileno()
        )


known_document_hashes = (
    load_document_hashes(
        GLOBAL_HASH_PATH
    )
)

print(
    "loaded document hash:",
    f"{len(known_document_hashes):,}",
)


# ============================================================
# 5. TEXT EXTRACTION
# ============================================================

def extract_text(row):
    preferred_fields = (
        "text",
        "content",
        "document",
        "body",
    )

    for field_name in preferred_fields:
        value = row.get(
            field_name
        )

        if isinstance(value, str):
            value = value.strip()

            if value:
                return value

    candidates = [
        value.strip()
        for value in row.values()
        if (
            isinstance(value, str)
            and len(value.strip()) >= 200
        )
    ]

    if not candidates:
        return ""

    return max(
        candidates,
        key=len,
    )


# ============================================================
# 6. TEXT QUALITY METRICS
# ============================================================

def calculate_text_statistics(text):
    character_count = len(text)

    letters = [
        character
        for character in text
        if character.isalpha()
    ]

    ascii_letter_count = sum(
        "a" <= character.lower() <= "z"
        for character in letters
    )

    if letters:
        ascii_letter_ratio = (
            ascii_letter_count
            / len(letters)
        )
    else:
        ascii_letter_ratio = 0.0

    printable_count = sum(
        character.isprintable()
        or character in "\n\t"
        for character in text
    )

    printable_ratio = (
        printable_count
        / max(character_count, 1)
    )

    lines = [
        line.strip()
        for line in text.splitlines()
        if line.strip()
    ]

    if lines:
        repeated_line_ratio = (
            1.0
            - len(set(lines)) / len(lines)
        )
    else:
        repeated_line_ratio = 0.0

    words = re.findall(
        r"\b[A-Za-z][A-Za-z'-]*\b",
        text,
    )

    sentence_count = len(
        re.findall(
            r"[.!?](?:\s|$)",
            text,
        )
    )

    symbol_count = sum(
        character in "{}[]<>`|\\"
        for character in text
    )

    symbol_ratio = (
        symbol_count
        / max(character_count, 1)
    )

    url_count = (
        text.lower().count("http://")
        + text.lower().count("https://")
        + text.lower().count("www.")
    )

    return {
        "characters": character_count,
        "words": len(words),
        "sentences": sentence_count,
        "ascii_letter_ratio": (
            ascii_letter_ratio
        ),
        "printable_ratio": (
            printable_ratio
        ),
        "repeated_line_ratio": (
            repeated_line_ratio
        ),
        "symbol_ratio": symbol_ratio,
        "url_count": url_count,
    }


def accept_document(
    text,
    domain,
):
    if len(text) < 500:
        return False

    statistics = (
        calculate_text_statistics(
            text
        )
    )

    if statistics["printable_ratio"] < 0.98:
        return False

    if statistics["repeated_line_ratio"] > 0.25:
        return False

    if domain == "math_science":
        if statistics["ascii_letter_ratio"] < 0.55:
            return False

        return True

    if statistics["ascii_letter_ratio"] < 0.90:
        return False

    if domain == "grammar":
        if len(text) < 1_000:
            return False

        if statistics["ascii_letter_ratio"] < 0.98:
            return False

        if statistics["repeated_line_ratio"] > 0.05:
            return False

        if statistics["symbol_ratio"] > 0.025:
            return False

        if statistics["url_count"] > 1:
            return False

        if statistics["sentences"] < 5:
            return False

        average_words_per_sentence = (
            statistics["words"]
            / max(
                statistics["sentences"],
                1,
            )
        )

        if not (
            6.0
            <= average_words_per_sentence
            <= 40.0
        ):
            return False

    return True


# ============================================================
# 7. SPLIT LONG TEXT
# ============================================================

def iterate_text_chunks(
    text,
    maximum_characters,
):
    text_length = len(text)
    start = 0

    while start < text_length:
        end = min(
            start + maximum_characters,
            text_length,
        )

        if end < text_length:
            newline_position = text.rfind(
                "\n",
                start,
                end,
            )

            sentence_position = text.rfind(
                ". ",
                start,
                end,
            )

            boundary = max(
                newline_position,
                sentence_position,
            )

            minimum_boundary = (
                start
                + maximum_characters // 2
            )

            if boundary >= minimum_boundary:
                end = boundary + 1

        chunk = text[
            start:end
        ].strip()

        if chunk:
            yield chunk

        start = end


# ============================================================
# 8. OPEN STREAMING DATASET
# ============================================================

def open_streaming_dataset(source):
    arguments = {
        "path": source["repo"],
        "split": source["split"],
        "streaming": True,
    }

    if source.get("config") is not None:
        arguments["name"] = source[
            "config"
        ]

    dataset = load_dataset(
        **arguments
    )

    shuffle_seed = source.get(
        "shuffle_seed"
    )

    if shuffle_seed is not None:
        dataset = dataset.shuffle(
            seed=int(shuffle_seed),
            buffer_size=10_000,
        )

    return dataset


def skip_iterator_rows(
    iterator,
    row_count,
    source_key,
):
    if row_count <= 0:
        return

    print(
        f"[{source_key}] "
        f"{row_count:,} row skipping..."
    )

    start_time = time.perf_counter()

    for index in range(row_count):
        try:
            next(iterator)
        except StopIteration as error:
            raise RuntimeError(
                f"{source_key}: data, skip "
                f"during ended. "
                f"Atlanan={index:,}/"
                f"{row_count:,}"
            ) from error

        completed = index + 1

        if (
            completed % 50_000 == 0
            or completed == row_count
        ):
            elapsed = (
                time.perf_counter()
                - start_time
            )

            print(
                f"[{source_key}] "
                f"skip={completed:,}/"
                f"{row_count:,} | "
                f"duration={elapsed / 60:.1f} dk"
            )


# ============================================================
# 9. SHARD YAZMA
# ============================================================

def write_uint16_shard(
    path,
    token_buffer,
):
    if not token_buffer:
        raise RuntimeError(
            "empty shard yazilamaz."
        )

    partial_path = path.with_suffix(
        path.suffix + ".partial"
    )

    if partial_path.exists():
        partial_path.unlink()

    with partial_path.open(
        "wb"
    ) as file:
        token_buffer.tofile(file)

        file.flush()
        os.fsync(
            file.fileno()
        )

    expected_bytes = (
        len(token_buffer)
        * 2
    )

    actual_bytes = (
        partial_path.stat().st_size
    )

    if actual_bytes != expected_bytes:
        raise RuntimeError(
            f"Shard size hatali: {partial_path}. "
            f"Expected={expected_bytes:,}, "
            f"found={actual_bytes:,}"
        )

    partial_path.replace(path)


# ============================================================
# 10. BUILD ONE SOURCE
# ============================================================

def build_streaming_source(
    source,
    known_hashes,
):
    source_key = source["key"]
    domain = source["domain"]

    target_tokens = int(
        source["target_tokens"]
    )

    source_root = (
        NEW_DATA_ROOT
        / source_key
    )

    source_root.mkdir(
        parents=True,
        exist_ok=True,
    )

    state_path = (
        source_root
        / "build_state.json"
    )

    manifest_path = (
        source_root
        / "manifest.json"
    )

    for partial_file in source_root.glob(
        "*.partial"
    ):
        print(
            "missing partial shard siliniyor:",
            partial_file,
        )

        partial_file.unlink()

    if state_path.exists():
        with state_path.open(
            "r",
            encoding="utf-8",
        ) as file:
            state = json.load(file)
    else:
        existing_binary_files = list(
            source_root.glob("*.bin")
        )

        if existing_binary_files:
            raise RuntimeError(
                f"{source_key}: state unavailable but "
                "binary shard found. Otomatik "
                "uzerine yazma yapilmadi."
            )

        state = {
            "version": 1,
            "source_key": source_key,
            "domain": domain,
            "target_tokens": target_tokens,
            "committed_tokens": 0,
            "rows_seen": 0,
            "accepted_documents": 0,
            "rejected_documents": 0,
            "duplicate_documents": 0,
            "shards": [],
            "complete": False,
        }

    if state.get("complete", False):
        if not manifest_path.exists():
            raise FileNotFoundError(
                f"{source_key}: completed as "
                "isaretli but manifest unavailable."
            )

        with manifest_path.open(
            "r",
            encoding="utf-8",
        ) as file:
            completed_manifest = json.load(file)

        print()
        print(
            f"[{source_key}] ZATEN complete | "
            f"token="
            f"{completed_manifest['tokens']:,}"
        )

        return completed_manifest

    committed_tokens = int(
        state["committed_tokens"]
    )

    rows_seen = int(
        state["rows_seen"]
    )

    accepted_documents = int(
        state["accepted_documents"]
    )

    rejected_documents = int(
        state["rejected_documents"]
    )

    duplicate_documents = int(
        state["duplicate_documents"]
    )

    shards = list(
        state["shards"]
    )

    for shard in shards:
        shard_path = Path(
            shard["path"]
        )

        if not shard_path.exists():
            raise FileNotFoundError(
                f"Registered shard is missing: {shard_path}"
            )

        expected_size = int(
            shard["tokens"]
        ) * 2

        actual_size = (
            shard_path.stat().st_size
        )

        if actual_size != expected_size:
            raise RuntimeError(
                f"Shard size bozuk: {shard_path}"
            )

    print()
    print("=" * 92)
    print(
        f"new data OLUSTURULUYOR: {source_key}"
    )
    print("=" * 92)

    print(
        "Domain          :",
        domain,
    )

    print(
        "target token     :",
        f"{target_tokens:,}",
    )

    print(
        "existing token    :",
        f"{committed_tokens:,}",
    )

    print(
        "resume satiri    :",
        f"{rows_seen:,}",
    )

    dataset = open_streaming_dataset(
        source
    )

    iterator = iter(dataset)

    initial_skip_rows = int(
        source.get(
            "skip_rows",
            0,
        )
    )

    total_skip_rows = (
        initial_skip_rows
        + rows_seen
    )

    skip_iterator_rows(
        iterator,
        total_skip_rows,
        source_key,
    )

    token_buffer = array("H")
    pending_hashes = []

    current_rows_seen = rows_seen
    start_time = time.perf_counter()
    last_report_time = start_time
    last_report_tokens = committed_tokens

    def commit_current_buffer():
        nonlocal token_buffer
        nonlocal pending_hashes
        nonlocal committed_tokens
        nonlocal shards
        nonlocal state
        nonlocal last_report_time
        nonlocal last_report_tokens

        if not token_buffer:
            return

        shard_index = len(shards)

        shard_path = (
            source_root
            / (
                f"{source_key}_"
                f"{shard_index:04d}.bin"
            )
        )

        write_uint16_shard(
            shard_path,
            token_buffer,
        )

        shard_tokens = len(
            token_buffer
        )

        committed_tokens += (
            shard_tokens
        )

        shard_record = {
            "path": str(shard_path),
            "tokens": shard_tokens,
            "bytes": shard_tokens * 2,
            "source": source_key,
            "domain": domain,
        }

        shards.append(
            shard_record
        )

        append_document_hashes(
            GLOBAL_HASH_PATH,
            pending_hashes,
        )

        pending_hashes = []

        state = {
            "version": 1,
            "source_key": source_key,
            "domain": domain,
            "target_tokens": target_tokens,
            "committed_tokens": (
                committed_tokens
            ),
            "rows_seen": current_rows_seen,
            "accepted_documents": (
                accepted_documents
            ),
            "rejected_documents": (
                rejected_documents
            ),
            "duplicate_documents": (
                duplicate_documents
            ),
            "shards": shards,
            "complete": (
                committed_tokens
                >= target_tokens
            ),
        }

        atomic_write_json(
            state_path,
            state,
        )

        current_time = (
            time.perf_counter()
        )

        interval_seconds = max(
            current_time
            - last_report_time,
            1e-9,
        )

        interval_tokens = (
            committed_tokens
            - last_report_tokens
        )

        encode_speed = (
            interval_tokens
            / interval_seconds
        )

        elapsed_minutes = (
            current_time
            - start_time
        ) / 60.0

        print(
            f"[{source_key}] "
            f"shard={shard_index:04d} | "
            f"token={committed_tokens:,}/"
            f"{target_tokens:,} | "
            f"rows={current_rows_seen:,} | "
            f"docs={accepted_documents:,} | "
            f"encode={encode_speed:,.0f} tok/s | "
            f"duration={elapsed_minutes:.1f} dk"
        )

        last_report_time = current_time
        last_report_tokens = (
            committed_tokens
        )

        token_buffer = array("H")

    while committed_tokens < target_tokens:
        try:
            row = next(iterator)
        except StopIteration as error:
            raise RuntimeError(
                f"{source_key}: data the-target "
                "before-reaching ended. "
                f"Token={committed_tokens:,}/"
                f"{target_tokens:,}"
            ) from error

        current_rows_seen += 1

        text = extract_text(
            row
        )

        if not accept_document(
            text,
            domain,
        ):
            rejected_documents += 1
            continue

        document_hash = hashlib.sha1(
            text.encode(
                "utf-8",
                errors="ignore",
            )
        ).hexdigest()

        if document_hash in known_hashes:
            duplicate_documents += 1
            continue

        remaining_tokens = (
            target_tokens
            - committed_tokens
            - len(token_buffer)
        )

        if remaining_tokens <= 0:
            commit_current_buffer()
            break

        document_added_tokens = 0

        for text_chunk in iterate_text_chunks(
            text,
            MAX_ENCODE_CHARACTERS,
        ):
            token_ids = tokenizer.encode(
                text_chunk,
                add_special_tokens=False,
            ).ids

            if not token_ids:
                continue

            remaining_tokens = (
                target_tokens
                - committed_tokens
                - len(token_buffer)
            )

            if remaining_tokens <= 0:
                break

            take_count = min(
                len(token_ids),
                remaining_tokens,
            )

            selected_ids = token_ids[
                :take_count
            ]

            if selected_ids:
                maximum_token_id = max(
                    selected_ids
                )

                if maximum_token_id >= 65_536:
                    raise RuntimeError(
                        "Token ID uint16 limit asti."
                    )

                token_buffer.extend(
                    selected_ids
                )

                document_added_tokens += (
                    take_count
                )

            if take_count < len(token_ids):
                break

        remaining_tokens = (
            target_tokens
            - committed_tokens
            - len(token_buffer)
        )

        if (
            document_added_tokens > 0
            and remaining_tokens > 0
        ):
            token_buffer.append(
                EOS_ID
            )

            document_added_tokens += 1

        if document_added_tokens == 0:
            rejected_documents += 1
            continue

        accepted_documents += 1
        known_hashes.add(
            document_hash
        )

        pending_hashes.append(
            document_hash
        )

        if (
            len(token_buffer)
            >= SHARD_TARGET_TOKENS
            or (
                committed_tokens
                + len(token_buffer)
                >= target_tokens
            )
        ):
            commit_current_buffer()

    if token_buffer:
        commit_current_buffer()

    if committed_tokens != target_tokens:
        raise RuntimeError(
            f"{source_key}: exact token hedefi "
            "tutulmadi. "
            f"found={committed_tokens:,}, "
            f"target={target_tokens:,}"
        )

    manifest = {
        "version": 1,
        "dtype": "uint16",
        "source": source_key,
        "domain": domain,
        "tokens": committed_tokens,
        "rows_seen": current_rows_seen,
        "accepted_documents": (
            accepted_documents
        ),
        "rejected_documents": (
            rejected_documents
        ),
        "duplicate_documents": (
            duplicate_documents
        ),
        "eos_id": EOS_ID,
        "vocab_size": (
            tokenizer.get_vocab_size()
        ),
        "shards": shards,
        "complete": True,
    }

    atomic_write_json(
        manifest_path,
        manifest,
    )

    state["complete"] = True

    atomic_write_json(
        state_path,
        state,
    )

    print(
        f"[{source_key}] completed | "
        f"token={committed_tokens:,} | "
        f"shard={len(shards):,}"
    )

    del dataset
    del iterator

    return manifest


# ============================================================
# 11. BUILD 400M NEW TOKENS
# ============================================================

streaming_types = {
    "streaming",
    "derived_streaming",
    "streaming_continuation",
}

streaming_sources = [
    source
    for source in training_plan[
        "data_mix"
    ]
    if source["dataset_type"]
    in streaming_types
]

expected_new_tokens = sum(
    int(source["target_tokens"])
    for source in streaming_sources
)

if expected_new_tokens != 400_000_000:
    raise RuntimeError(
        "new streaming data total "
        "400M token not. "
        f"found={expected_new_tokens:,}"
    )

source_manifests = []

for source in streaming_sources:
    source_manifest = (
        build_streaming_source(
            source,
            known_document_hashes,
        )
    )

    source_manifests.append(
        source_manifest
    )


# ============================================================
# 12. GLOBAL NEW-DATA MANIFEST
# ============================================================

actual_new_tokens = sum(
    int(manifest["tokens"])
    for manifest in source_manifests
)

if actual_new_tokens != 400_000_000:
    raise RuntimeError(
        "generated new data 400M token not. "
        f"found={actual_new_tokens:,}"
    )

all_new_shards = []

for manifest in source_manifests:
    all_new_shards.extend(
        manifest["shards"]
    )

new_data_manifest = {
    "version": 1,
    "dtype": "uint16",
    "vocab_size": (
        tokenizer.get_vocab_size()
    ),
    "eos_id": EOS_ID,
    "tokens": actual_new_tokens,
    "sources": source_manifests,
    "shards": all_new_shards,
    "complete": True,
}

atomic_write_json(
    NEW_MANIFEST_PATH,
    new_data_manifest,
)


# ============================================================
# 13. RESULT
# ============================================================

print()
print("=" * 100)
print("STAGE-2A new data ready")
print("=" * 100)

for manifest in source_manifests:
    print(
        f"{manifest['source']:<28} | "
        f"domain={manifest['domain']:<22} | "
        f"token={manifest['tokens']:>12,} | "
        f"shard={len(manifest['shards']):>3}"
    )

print("-" * 100)

print(
    "Total new token:",
    f"{actual_new_tokens:,}",
)

print(
    "Total shard     :",
    len(all_new_shards),
)

print(
    "Manifest         :",
    NEW_MANIFEST_PATH,
)

print()
print(
    "400M new TOKEN HAZIRLAMA: PASS"
)